# Lab 4. VQE — 회로로 바닥상태를 찾고, 결과를 검증하기

## 실습 소개

이번 Lab에서는 <strong>양자 회로가 준비한 상태의 에너지를 계산하고, 회로의 각도를 바꾸어 가장 낮은 에너지의 상태를 찾아봅니다.</strong> 탐색이 끝나면 에너지와 준비된 상태를 함께 확인하고, 결과가 충분히 좋지 않다면 무엇을 바꾸어야 하는지 판단합니다.

Lab3에서는 저장한 논리 정보를 오류로부터 복원했습니다. 이번에는 원하는 상태를 찾아 준비하기 위해, 상태를 평가할 기준으로 에너지를 사용합니다. 양자계의 에너지를 나타내는 연산자를 <strong>Hamiltonian, 또는 해밀토니안</strong>이라고 하며, 여기서는 Hadamard 게이트 $H$와 구분해 $\mathcal H$로 쓰겠습니다.

상태가 $|\psi\rangle$일 때 에너지 기댓값은

$$
E=\langle\psi|\mathcal H|\psi\rangle
$$

입니다. 같은 상태를 반복해서 준비하고 에너지를 측정했을 때 얻는 평균으로, 일반적인 상태의 측정 결과가 매번 같다는 뜻은 아닙니다. 이 Lab에서 “상태의 에너지”는 이 기댓값을 가리킵니다.

Hamiltonian의 고유상태 $|E_k\rangle$는 $\mathcal H|E_k\rangle=E_k|E_k\rangle$를 만족하고, 이 상태에서 에너지를 측정하면 $E_k$가 나옵니다. 가능한 고유값 중 가장 작은 값이 <strong>바닥 에너지</strong> $E_0$이며, 그 값에 대응하는 상태를 <strong>바닥상태(ground state)</strong>라고 부릅니다.

정규화된 상태를 고유상태들의 중첩으로 쓰면

$$
|\psi\rangle=\sum_k c_k|E_k\rangle,
\qquad \sum_k|c_k|^2=1,
\qquad E=\sum_k|c_k|^2E_k\ge E_0.
$$

에너지는 고유값들의 확률 가중 평균이므로 가장 작은 고유값보다 낮아질 수 없습니다. 이것이 이번 실습에서 사용하는 <strong>변분 원리</strong>입니다. 에너지를 낮추며 바닥상태를 찾되, 마지막에는 얻은 에너지와 상태가 목표에 얼마나 가까운지 확인하겠습니다.

서로 다른 상태를 시험하기 위해 회전각을 바꿀 수 있는 회로를 사용합니다. 회로 구조를 정해 두고 매개변수로 탐색할 상태들을 정하는 방식을 <strong>ansatz</strong>라고 합니다. <strong>회로 구조는 준비할 수 있는 상태의 범위를 정하고, 각도는 그 범위 안에서 상태 하나를 선택합니다.</strong> 각도를 바꿀 수 있다고 해서 모든 상태를 만들 수 있는 것은 아닙니다.

고전 최적화기는 각도를 입력받아 에너지 하나를 반환하는 <strong>목적함수</strong>를 반복해서 호출합니다. 전체 과정은 <strong>각도 선택 → 상태 준비 → 에너지 평가 → 각도 갱신 → 결과 검증</strong>입니다. 이렇게 매개변수 양자 회로와 고전 최적화를 함께 사용해 바닥상태를 찾는 방법을 <strong>VQE(Variational Quantum Eigensolver)</strong>라고 합니다. 다음 각도를 시험할 때에는 상태 준비부터 다시 시작하며, 측정이 끝난 상태에 계속 게이트를 붙이는 절차가 아닙니다.

이번에 사용할 두 큐비트 Hamiltonian과 비교 기준은 다음과 같습니다.

$$
\mathcal H=-Z_0Z_1-X_0Y_1-\frac12(Z_0+Z_1),
\qquad E_0=-1-\sqrt2\approx-2.4142.
$$

아래첨자는 연산자가 작용하는 큐비트 번호입니다. 각 항을 어떻게 측정하고 에너지로 합하는지는 문제 1에서 설명합니다. 이 작은 문제는 행렬의 고유값을 직접 계산할 수 있어 정확한 바닥 에너지를 비교 기준으로 제공합니다. 최적화기는 정답 각도나 바닥상태를 전달받지 않고, 여러분의 목적함수가 반환하는 에너지를 이용합니다.

복습자료에서는 $R_y$와 CX를 사용하는 회로로 VQE를 실행하고, CX를 제거한 경우를 비교했습니다. 이번에는 익숙한 회로에서 출발해 <strong>새 Hamiltonian의 바닥상태도 그 회로로 준비할 수 있는지</strong> 살펴봅니다. 필요한 경우 회로를 확장하고, 개선된 이유를 상태식과 연결합니다.

| 문제 | 확인할 내용 |
|---|---|
| <strong>1. 측정값에서 에너지 계산하기</strong> | 각 관측량을 읽고, 기댓값들을 하나의 에너지로 합칩니다. |
| <strong>2. 회로가 탐색할 수 있는 상태 확인하기</strong> | 첫 회로의 상태와 에너지 식으로 도달할 수 있는 범위를 판단합니다. |
| <strong>3. 회로를 확장하고 최적화하기</strong> | 필요한 상태를 준비하도록 회로를 수정하고 목적함수에 연결합니다. |
| <strong>4. 결과를 검증하고 원인 진단하기</strong> | 종료 메시지, 정확한 에너지, 상태 일치도와 유한 shots 결과를 구분합니다. |

마지막에는 Hamiltonian의 항 하나의 부호를 바꾸어, 앞의 회로와 매개변수를 어떻게 활용할지 판단합니다. <strong>최적화기가 종료했다는 사실과 바닥상태를 찾았다는 결론은 구분</strong>해야 합니다. 실제 결과를 근거로 회로의 표현 범위, 탐색 과정, 측정의 영향을 차례로 확인하겠습니다.

---

## 노트북 사용 방법

각 절은 계산의 목적과 기호 정의, 필요한 수식과 예제, 직접 풀 문제로 이어집니다. 회로가 준비하는 상태와 그 상태를 평가하는 계산이 어떻게 연결되는지 확인하며 읽으세요.

코드 셀은 위에서 아래로 실행합니다. ‘제공 코드’에는 상태·기댓값 계산, 최적화, 측정 기록 생성과 표 출력 도구가 들어 있으므로 이 셀도 실행해야 합니다. 내부 구현을 모두 분석할 필요는 없으며, 각 문제에서 필요한 사용법을 안내합니다.

1. <strong>계산과 예상을 먼저 작성합니다.</strong> Hamiltonian, 회로 구조, 바꿀 각도를 확인하고 결과를 예상합니다.
2. <strong>TODO를 완성하고 검사합니다.</strong> `TODO BEGIN`과 `TODO END` 사이를 작성하되 함수 이름과 매개변수는 유지합니다. 답안 셀 다음에 공개 확인 셀을 실행하세요.
3. <strong>예상 답안을 작성한 뒤 해당 실습 코드를 실행합니다.</strong> 학생용의 `RUN_OBSERVATIONS`는 처음에 `False`입니다. 예상을 적은 뒤 `True`로 바꾸고 설정 셀과 해당 문항의 실습 코드 셀을 실행하세요. 다음 문항도 결과를 열기 전에 먼저 예상합니다. 실습 코드가 다시 실행되지 않도록 하려면 `False`로 바꾸고 설정 셀을 실행합니다. 공개 함수 검사는 이 설정과 무관하게 작동합니다.
4. <strong>실행 결과와 근거를 덧붙입니다.</strong> 코드를 수정했다면 답안·검사·연결 실험을 다시 실행합니다. 회로가 바뀌었으면 이전 최적화 결과도 다시 계산하세요.

| 작성할 역할 | 입력과 출력 |
|---|---|
| 상태 준비 회로 | 각도에서 회로를 만듭니다. |
| 에너지 평가기 | 관측량의 기댓값들을 에너지로 변환합니다. |
| 최적화 목적함수 | 회로 준비와 에너지 평가를 연결해 에너지 하나를 반환합니다. |

최적화기는 제공됩니다. 여러분의 목적함수는 매번 입력받은 각도를 사용해야 하며, 알고 있는 정답 에너지를 직접 반환해서는 안 됩니다. 마지막 적용은 예상값과 제안 각도를 입력한 뒤 별도의 `RUN_APPLICATION`을 켜서 확인합니다.

회전각은 <strong>라디안</strong>입니다. Qiskit의 상태벡터 기저와 Pauli 문자열은 두 큐비트에서 $q_1q_0$ 순서이므로, 수식의 큐비트 첨자를 옮길 때 문제의 대응표를 확인하세요.

앞부분은 이상적인 상태의 <strong>정확한 기댓값</strong>을 사용합니다. 뒤에서는 정해진 횟수만큼 새 상태를 준비하고 측정한 기록으로 에너지를 추정합니다. 이 반복 횟수가 <strong>shots</strong>이며, 유한한 표본에는 변동이 있습니다. 정확값과 추정값을 구분하고, shots가 측정 설정별 횟수인지 전체 횟수인지 확인하세요.

확인 셀의 <strong>미완성</strong>은 남은 TODO, <strong>실패</strong>는 실행 오류 또는 조건 불일치, <strong>통과</strong>는 공개 코드 검사 통과를 뜻합니다. 서술 답안은 별도로 작성합니다. 에너지가 낮아졌다는 결과, 주어진 회로에서 최솟값을 찾았다는 판단, 실제 바닥상태에 도달했다는 판단을 결과와 근거로 구분하세요.

이 Lab에서는 <strong>AI를 활용해 공부해도 좋습니다.</strong> 개념 설명, 계산 점검, 디버깅에 활용하되 제안받은 코드도 직접 실행하세요. 설명 문항에는 실행 결과를 근거로 회로 수정과 검증이 왜 적절한지 자신의 말로 작성합니다.

## 실행 환경 준비

이 노트북은 Python 3.12, Qiskit 2.3.1, NumPy 2.2.6, SciPy 1.18.1에서 검증했습니다. 로컬에서는 함께 배포된 `requirements.txt`를 설치한 환경을 Jupyter 커널로 선택하세요.

패키지가 없다면 새 코드 셀에서 다음 명령을 실행한 뒤 커널을 다시 시작하고, 노트북을 위에서부터 실행하세요.

```python
%pip install qiskit==2.3.1 numpy==2.2.6 scipy==1.18.1
```

아래 제공 도구 셀도 실행합니다. 긴 코드는 접혀 보일 수 있습니다. 각 문제에서 도구의 역할과 사용법을 안내하므로 내부 구현을 모두 분석한 뒤 시작할 필요는 없습니다.

In [ ]:
import sys
from importlib.metadata import version
print("Python", sys.version.split()[0])
for package in ("qiskit", "numpy", "scipy"):
    print(package, version(package))

### 제공 도구와 실습 코드 실행 설정

아래 도구 셀을 실행하세요. 공개 확인 셀은 학습용 검사이며 서술 답안은 별도로 작성합니다. 실행 결과를 보기 전에는 각 문항의 예상과 계산을 먼저 적습니다.

학생용은 `RUN_OBSERVATIONS=False`로 시작합니다. 예상 답안 작성을 마치면 아래 설정을 `True`로 바꾸어 실행하고, 해당 문항의 결과 확인용 코드 셀을 실행하세요. 이 설정은 함수 TODO가 아닙니다. 마지막 적용에는 별도의 실행 설정이 있습니다.

In [ ]:
"""제공 코드: 두 큐비트 VQE의 상태·측정·최적화·공개 확인 도구."""
import numbers
from html import escape
from types import CodeType, FunctionType, ModuleType
from contextlib import contextmanager
import numpy as np
from scipy.optimize import minimize
from IPython.display import HTML, display
from qiskit import QuantumCircuit
from qiskit.circuit import Gate, Instruction
from qiskit.quantum_info import Operator, SparsePauliOp, Statevector, state_fidelity

ATOL = 1e-9
ENERGY_TOL = 1e-6
FIDELITY_TOL = 1e-6
OBSERVABLE_KEYS = ('ZZ', 'YX', 'IZ', 'ZI')
OBSERVABLES = {key: SparsePauliOp(key) for key in OBSERVABLE_KEYS}
CHECK_LABELS = ('1b · 에너지 평가기', '3a · 상대위상 회로', '3b · 최적화 목적함수')
CHECK_STATUS = {}
LAB_RESULTS = {}


def show_table(headers, rows):
    def cell_text(value):
        if isinstance(value, (float, np.floating)):
            return f'{value:.8f}'
        if isinstance(value, (tuple, list)):
            return '(' + ', '.join(cell_text(x) for x in value) + ')'
        return str(value)
    rows = list(rows)
    if any(len(row) != len(headers) for row in rows):
        raise ValueError('표의 열 수를 확인하세요.')
    heading = ''.join(f'<th>{escape(str(x))}</th>' for x in headers)
    body = ''.join('<tr>' + ''.join(f'<td>{escape(cell_text(x))}</td>' for x in row) + '</tr>' for row in rows)
    display(HTML(f'<table><thead><tr>{heading}</tr></thead><tbody>{body}</tbody></table>'))


def _real(value, label='반환값'):
    if not isinstance(value, numbers.Real) or isinstance(value, (bool, np.bool_)) or not np.isfinite(value):
        raise AssertionError(f'{label}: 유한한 실수를 사용하세요.')
    return float(value)


def _positive_integer(value, label):
    if not isinstance(value, (int, np.integer)) or isinstance(value, (bool, np.bool_)) or value < 1:
        raise ValueError(f'{label}: 양의 정수가 필요합니다.')
    return int(value)


def hamiltonian(xy_coeff=-1.0):
    coefficient = _real(xy_coeff, 'XY 계수')
    return SparsePauliOp.from_list([('ZZ', -1.0), ('YX', coefficient), ('IZ', -0.5), ('ZI', -0.5)])


@contextmanager
def _instrument_dependencies(function, replacements):
    """실제 호출 경로의 함수·별칭을 잠시 계측하고 모든 참조를 복원합니다."""
    if not isinstance(function, FunctionType):
        raise AssertionError('제공된 함수 정의 안에 답안을 작성하세요.')
    namespaces, functions, originals = {}, {}, {}
    library_prefixes = ('numpy', 'scipy', 'qiskit', 'IPython', 'importlib', 'contextlib')

    def code_names(code):
        names = set(code.co_names)
        for constant in code.co_consts:
            if isinstance(constant, CodeType):
                names.update(code_names(constant))
        return names

    def dependency(value, name=None):
        key = name if name in replacements else getattr(value, '__name__', None)
        if isinstance(value, FunctionType) and key in replacements:
            originals[id(value)] = (value, replacements[key])
            return True
        return False

    def visit(candidate):
        if not isinstance(candidate, FunctionType) or id(candidate) in functions:
            return
        if dependency(candidate):
            return
        if (candidate.__module__ or '').startswith(library_prefixes):
            return
        functions[id(candidate)] = candidate
        namespace = candidate.__globals__
        namespaces[id(namespace)] = namespace
        for name in replacements:
            dependency(namespace.get(name), name)
        names = code_names(candidate.__code__)
        for name in names:
            value = namespace.get(name)
            if isinstance(value, FunctionType):
                if not dependency(value, name):
                    visit(value)
            elif isinstance(value, ModuleType) and not value.__name__.startswith(library_prefixes):
                module_namespace = vars(value)
                namespaces[id(module_namespace)] = module_namespace
                for attribute in names:
                    target = module_namespace.get(attribute)
                    if not dependency(target, attribute):
                        visit(target)
        for value in (*tuple(candidate.__defaults__ or ()),
                      *tuple((candidate.__kwdefaults__ or {}).values())):
            if not dependency(value):
                visit(value)
        for cell in candidate.__closure__ or ():
            try:
                value = cell.cell_contents
            except ValueError:
                continue
            if not dependency(value):
                visit(value)

    visit(function)
    saved_bindings, saved_defaults, saved_cells = [], [], []
    seen_cells = set()

    def substituted(value):
        match = originals.get(id(value))
        return match[1] if match is not None and value is match[0] else value

    try:
        for namespace in namespaces.values():
            for name, value in tuple(namespace.items()):
                replacement = substituted(value)
                if replacement is not value:
                    saved_bindings.append((namespace, name, value))
                    namespace[name] = replacement
        for candidate in functions.values():
            defaults, keyword_defaults = candidate.__defaults__, candidate.__kwdefaults__
            new_defaults = None if defaults is None else tuple(substituted(value) for value in defaults)
            new_keywords = None if keyword_defaults is None else {
                name: substituted(value) for name, value in keyword_defaults.items()}
            if (defaults is not None and any(a is not b for a, b in zip(defaults, new_defaults))) or (
                    keyword_defaults is not None and any(keyword_defaults[key] is not new_keywords[key] for key in keyword_defaults)):
                saved_defaults.append((candidate, defaults, keyword_defaults))
                candidate.__defaults__, candidate.__kwdefaults__ = new_defaults, new_keywords
            for cell in candidate.__closure__ or ():
                if id(cell) in seen_cells:
                    continue
                seen_cells.add(id(cell))
                try:
                    value = cell.cell_contents
                except ValueError:
                    continue
                replacement = substituted(value)
                if replacement is not value:
                    saved_cells.append((cell, value))
                    cell.cell_contents = replacement
        yield
    finally:
        for cell, value in reversed(saved_cells):
            cell.cell_contents = value
        for candidate, defaults, keyword_defaults in reversed(saved_defaults):
            candidate.__defaults__, candidate.__kwdefaults__ = defaults, keyword_defaults
        for namespace, name, value in reversed(saved_bindings):
            namespace[name] = value


def _check_circuit_operations(circuit, allowed=None, active=None):
    """허용된 기본 게이트와 그 게이트만 묶은 사용자 정의 회로를 검사합니다."""
    forbidden = {'measure', 'reset', 'initialize', 'set_statevector', 'set_density_matrix',
                 'if_else', 'while_loop', 'for_loop', 'switch_case'}
    active = set() if active is None else active
    if id(circuit) in active:
        raise AssertionError('순환하는 합성 게이트 정의는 사용할 수 없습니다.')
    active.add(id(circuit))
    try:
        if circuit.num_clbits:
            raise AssertionError('상태 준비 회로에는 고전 비트를 넣지 마세요.')
        for item in circuit.data:
            operation = item.operation
            if operation.name in forbidden or operation.num_clbits:
                raise AssertionError('측정·초기화·reset·조건부 제어 없이 유니터리 게이트를 사용하세요.')
            if allowed is not None and operation.name in allowed:
                continue
            # to_gate/to_instruction으로 묶은 회로만 펼칩니다. 다른 기본 게이트의
            # 분해를 이용해 이 문항의 허용 게이트 조건을 우회하지 않습니다.
            if operation.base_class in (Gate, Instruction) and operation.definition is not None:
                _check_circuit_operations(operation.definition, allowed, active)
            elif allowed is not None:
                raise AssertionError('해당 함수에 허용된 기본 게이트로 회로를 작성하세요. 합성 게이트도 내부를 검사합니다.')
    finally:
        active.remove(id(circuit))


def circuit_state(circuit, require_ansatz=False):
    if not isinstance(circuit, QuantumCircuit):
        raise AssertionError('QuantumCircuit을 반환하세요.')
    if (circuit.num_qubits, circuit.num_clbits) != (2, 0) or circuit.parameters:
        raise AssertionError('수치 각도가 정해진 2큐비트·고전 비트 0개 회로가 필요합니다.')
    allowed = {'ry', 'rz', 'cx', 'barrier'} if require_ansatz else None
    _check_circuit_operations(circuit, allowed)
    matrix = np.asarray(Operator(circuit).data)
    if not np.isfinite(matrix).all() or not np.allclose(np.einsum('ji,jk->ik', matrix.conj(), matrix), np.eye(4), atol=ATOL, rtol=0):
        raise AssertionError('유한한 유니터리 회로가 필요합니다.')
    return Statevector.from_instruction(circuit)


def _expectations_from_state(state):
    return {key: float(np.real(state.expectation_value(operator))) for key, operator in OBSERVABLES.items()}


def expectations_from_circuit(circuit):
    """같은 회로 상태의 정확한 ZZ, YX, IZ, ZI 기댓값을 반환합니다."""
    return _expectations_from_state(circuit_state(circuit))


def exact_energy(circuit, xy_coeff=-1.0):
    """검증용: Hamiltonian 연산자로 직접 계산합니다."""
    return float(np.real(circuit_state(circuit).expectation_value(hamiltonian(xy_coeff))))


def ground_reference(xy_coeff=-1.0):
    """검증 단계에 사용하는 정확한 고유값과 바닥상태입니다."""
    eigenvalues, eigenvectors = np.linalg.eigh(hamiltonian(xy_coeff).to_matrix())
    return {'energy': float(eigenvalues[0]), 'state': Statevector(eigenvectors[:, 0]),
            'eigenvalues': tuple(map(float, eigenvalues))}


def validate_result(circuit, xy_coeff=-1.0):
    state = circuit_state(circuit)
    reference = ground_reference(xy_coeff)
    energy = float(np.real(state.expectation_value(hamiltonian(xy_coeff))))
    delta = energy - reference['energy']
    fidelity = float(state_fidelity(state, reference['state']))
    return {'energy': energy, 'reference_energy': reference['energy'], 'delta_energy': delta,
            'fidelity': fidelity, 'energy_pass': abs(delta) <= ENERGY_TOL,
            'state_pass': fidelity >= 1 - FIDELITY_TOL}


def real_ansatz(theta):
    circuit = QuantumCircuit(2)
    circuit.ry(_real(theta, 'theta'), 0)
    circuit.cx(0, 1)
    return circuit


def phase_before_rotation(theta, phi):
    """3(c)의 비교 회로: 초기 RZ, RY, CX 순서입니다."""
    circuit = QuantumCircuit(2)
    circuit.rz(_real(phi, 'phi'), 0)
    circuit.ry(_real(theta, 'theta'), 0)
    circuit.cx(0, 1)
    return circuit


def measurement_basis(setting):
    circuit = QuantumCircuit(2)
    if setting == 'XY':
        circuit.h(0)
        circuit.sdg(1)
        circuit.h(1)
    elif setting != 'Z':
        raise ValueError('측정 설정은 Z 또는 XY입니다.')
    return circuit


def basis_probabilities(circuit, setting):
    state = circuit_state(circuit).evolve(measurement_basis(setting))
    probabilities = np.asarray(state.probabilities(), dtype=float)
    return probabilities / probabilities.sum()


def _counts_vector(counts):
    if not isinstance(counts, dict) or not counts:
        raise ValueError('두 비트 문자열과 정수 횟수의 dict가 필요합니다.')
    if any(key not in {'00', '01', '10', '11'} for key in counts):
        raise ValueError('측정 문자열은 q1q0 순서의 두 비트입니다.')
    if any(not isinstance(n, (int, np.integer)) or isinstance(n, (bool, np.bool_)) or n < 0 for n in counts.values()):
        raise ValueError('측정 횟수는 음이 아닌 정수입니다.')
    vector = np.array([counts.get(f'{i:02b}', 0) for i in range(4)], dtype=float)
    if vector.sum() <= 0:
        raise ValueError('측정 횟수의 합이 양수여야 합니다.')
    return vector / vector.sum()


def terms_from_counts(z_counts, xy_counts):
    """Z 기록의 세 통계와 별도의 XY 기록을 각 설정 안에서 평균합니다."""
    z_probs = _counts_vector(z_counts)
    xy_probs = _counts_vector(xy_counts)
    q0 = np.array([1, -1, 1, -1])
    q1 = np.array([1, 1, -1, -1])
    parity = q0 * q1
    return {'ZZ': float(z_probs @ parity), 'YX': float(xy_probs @ parity),
            'IZ': float(z_probs @ q0), 'ZI': float(z_probs @ q1)}


def sample_energy(circuit, energy_function, shots, seed=None, xy_coeff=-1.0):
    shots = _positive_integer(shots, '설정별 shots')
    streams = np.random.SeedSequence(seed).spawn(2)
    records = {}
    for setting, stream in zip(('Z', 'XY'), streams):
        values = np.random.default_rng(stream).multinomial(shots, basis_probabilities(circuit, setting))
        records[setting] = {f'{index:02b}': int(count) for index, count in enumerate(values) if count}
    terms = terms_from_counts(records['Z'], records['XY'])
    energy = _real(energy_function(terms, xy_coeff=xy_coeff), '에너지 추정값')
    return {'energy': energy, 'expectations': terms, 'counts': records,
            'shots_per_setting': shots, 'total_shots': 2 * shots}


def repeated_energy_estimates(circuit, energy_function, shots_values=(64, 256, 1024), repeats=20, seed=20261109, xy_coeff=-1.0):
    repeats = _positive_integer(repeats, '반복 횟수')
    if repeats < 2:
        raise ValueError('표본 표준편차를 계산하려면 두 번 이상 반복합니다.')
    shots_values = tuple(_positive_integer(value, '설정별 shots') for value in shots_values)
    streams = np.random.SeedSequence(seed).spawn(len(shots_values) * repeats)
    rows = []
    exact = exact_energy(circuit, xy_coeff)
    for group, shots in enumerate(shots_values):
        values = []
        for index in range(repeats):
            local_seed = int(streams[group * repeats + index].generate_state(1)[0])
            trial = sample_energy(circuit, energy_function, shots, local_seed, xy_coeff)
            values.append(trial['energy'])
        rows.append({'shots_per_setting': shots, 'repeats': repeats,
                     'total_shots_per_estimate': 2 * shots, 'total_shots': 2 * shots * repeats,
                     'energies': values, 'exact_energy': exact,
                     'mean': float(np.mean(values)), 'sample_std': float(np.std(values, ddof=1)),
                     'min': min(values), 'max': max(values)})
    return rows


def optimize_vqe(objective, initial=(0.4, 0.6)):
    initial = np.array([_real(x, '초기 각도') for x in initial], dtype=float)
    if initial.shape != (2,):
        raise ValueError('초기값은 theta,phi 두 값입니다.')
    history = []
    def observed(parameters):
        point = tuple(map(float, parameters))
        energy = _real(objective(np.asarray(parameters).copy()), '목적함수 반환값')
        history.append({'parameters': point, 'energy': energy})
        return energy
    result = minimize(observed, initial, method='BFGS', options={'gtol': 1e-6, 'maxiter': 120})
    return {'parameters': tuple(map(float, result.x)), 'objective_energy': float(result.fun),
            'nfev': int(result.nfev), 'nit': int(result.nit), 'success': bool(result.success),
            'message': str(result.message), 'initial': tuple(map(float, initial)), 'history': history}


def _run_check(label, check):
    try:
        check()
    except NotImplementedError:
        CHECK_STATUS[label] = '미완성'
        print(f'{label}: 미완성 — TODO를 작성한 뒤 다시 실행하세요.')
        return False
    except Exception as error:
        CHECK_STATUS[label] = '실패'
        print(f'{label}: 실패 — {type(error).__name__}: {error}')
        return False
    CHECK_STATUS[label] = '통과'
    print(f'{label}: 통과')
    return True


def check_energy_from_expectations(function):
    def check():
        for vector in ([1, 0, 0, 0], [1, 2j, -0.7, 0.3j], [0.4j, -0.8, 1.1j, 0.6]):
            vector = np.asarray(vector, dtype=complex)
            state = Statevector(vector / np.linalg.norm(vector))
            values = _expectations_from_state(state)
            for coefficient in (-1.0, 1.0, 0.37):
                actual = _real(function(dict(values), xy_coeff=coefficient))
                expected = float(np.real(state.expectation_value(hamiltonian(coefficient))))
                assert np.isclose(actual, expected, atol=ATOL, rtol=0), '항의 계수·부호와 딕셔너리 키를 확인하세요.'
        estimated = {'ZZ': 0.9, 'YX': 0.8, 'IZ': 0.9, 'ZI': 0.9}
        for coefficient, expected in ((-1.0, -2.6), (1.0, -1.0), (0.5, -1.4)):
            actual = _real(function(dict(estimated), xy_coeff=coefficient))
            assert np.isclose(actual, expected, atol=ATOL, rtol=0), '표본 추정값도 입력된 수치와 계수로 계산하세요.'
    return _run_check(CHECK_LABELS[0], check)


def check_phase_ansatz(function):
    def check():
        for theta, phi in ((0.0, 0.8), (0.43, -0.79), (1.31, 2.17), (-0.63, 0.92)):
            state = circuit_state(function(theta, phi), require_ansatz=True)
            target = Statevector([np.cos(theta / 2), 0, 0, np.exp(1j * phi) * np.sin(theta / 2)])
            assert state_fidelity(state, target) >= 1 - ATOL, '목표 상태의 진폭과 상대위상을 확인하세요. 전역위상 차이는 허용합니다.'
    return _run_check(CHECK_LABELS[1], check)


def _check_objective_wiring(function):
    """답안은 그대로 두고, 호출 도구의 반환값을 이어 사용하는지 확인합니다."""
    if not isinstance(function, FunctionType):
        raise AssertionError('제공된 함수 정의 안에 목적함수를 작성하세요.')
    point = (0.37, -0.58)
    probe = QuantumCircuit(2)
    probe.ry(0.271, 0)
    probe.ry(-0.38, 1)
    probe.cx(1, 0)
    values = {'ZZ': 0.12, 'YX': -0.27, 'IZ': 0.08, 'ZI': 0.31}
    marker = -0.76329
    calls = {'phase': 0, 'expectations': 0, 'energy': 0}
    def phase(theta, phi):
        assert np.allclose([theta, phi], point, atol=ATOL, rtol=0), '입력 theta,phi 순서를 확인하세요.'
        calls['phase'] += 1
        return probe
    def expectations(circuit):
        assert state_fidelity(circuit_state(circuit), circuit_state(probe)) >= 1 - ATOL, '자신의 phase_ansatz가 반환한 회로를 사용하세요.'
        calls['expectations'] += 1
        return dict(values)
    def energy(received, xy_coeff=-1.0):
        assert set(received) == set(values) and all(np.isclose(received[k], values[k], atol=ATOL, rtol=0) for k in values), '제공 도구의 기댓값을 에너지 평가기에 전달하세요.'
        assert xy_coeff == -1.0, '기본 Hamiltonian의 XY 계수를 사용하세요.'
        calls['energy'] += 1
        return marker
    replacements = {'phase_ansatz': phase, 'expectations_from_circuit': expectations,
                    'energy_from_expectations': energy}
    with _instrument_dependencies(function, replacements):
        actual = _real(function(np.array(point)))
        assert all(calls.values()), '회로·기댓값 도구·에너지 평가기를 모두 연결하세요.'
        assert np.isclose(actual, marker, atol=ATOL, rtol=0), '자신의 에너지 평가기가 반환한 값을 목적함수에서 반환하세요.'
    return calls



def check_vqe_objective(function):
    def check():
        for point in ((0.4, 0.6), (1.2, -0.7), (-0.8, 1.4)):
            actual = _real(function(np.array(point)))
            state = Statevector([np.cos(point[0] / 2), 0, 0, np.exp(1j * point[1]) * np.sin(point[0] / 2)])
            expected = float(np.real(state.expectation_value(hamiltonian())))
            assert np.isclose(actual, expected, atol=ATOL, rtol=0), '입력 각도에서의 회로와 에너지 연결을 확인하세요.'
        _check_objective_wiring(function)
    return _run_check(CHECK_LABELS[2], check)


def run_demo(label, experiment, requires=()):
    LAB_RESULTS.pop(label, None)
    missing = [key for key in requires if CHECK_STATUS.get(key) != '통과']
    if missing:
        print(f'[대기] {label}: 먼저 다음 코드 확인을 통과하세요: {", ".join(missing)}')
        return None
    try:
        result = experiment()
    except NotImplementedError:
        print(f'[대기] {label}: 앞의 TODO를 완성하세요.')
        return None
    except Exception as error:
        print(f'[실험 오류] {label}: {type(error).__name__}: {error}')
        return None
    LAB_RESULTS[label] = result
    return result

In [ ]:
RUN_OBSERVATIONS = False
print("관찰 실험 실행:", RUN_OBSERVATIONS)
print("학생용에서는 먼저 예상과 계산을 적은 뒤 True로 바꾸고 연결 실험 셀들을 실행하세요.")

## 문제 1. 측정한 값들을 어떻게 에너지로 바꿀까요?

이번 문제에서는 <strong>회로가 준비한 상태를 에너지 하나로 평가하는 방법</strong>을 만듭니다. 뒤에서 고전 최적화기는 여러 각도를 시도하며 이 에너지를 비교합니다. 따라서 먼저 어떤 관측량을 구해야 하는지, 각 결과를 어떤 부호와 계수로 합해야 하는지 확인해야 합니다.

이번 실습의 Hamiltonian은

$$
\mathcal H
=
-Z_0Z_1-X_0Y_1-\frac12(Z_0+Z_1)
$$

입니다. 이를 구성하는 네 항의 기댓값을 구하면, 같은 계수로 합하여 상태의 에너지를 계산할 수 있습니다.

---

### Hamiltonian의 각 항 읽기

먼저 아래첨자와 곱의 의미를 구분하겠습니다.

$Z_0$는 q0의 Z 관측량입니다. 두 큐비트 전체에 대한 연산자로 보면 q0에는 Z가, q1에는 항등 연산이 작용합니다. $Z_1$은 반대로 q1의 Z 관측량입니다.

$Z_0Z_1$은 두 큐비트의 Z 관측값을 곱한 공동 관측량입니다. 두 큐비트를 같은 시행에서 Z 측정한 뒤, 각각의 결과를 $+1$ 또는 $-1$로 읽어 곱합니다. 두 관측값이 같으면 곱은 $+1$, 다르면 $-1$입니다.

$X_0Y_1$도 같은 방식으로 읽습니다. q0는 X 방향으로, q1은 Y 방향으로 측정하고, <strong>같은 시행에서 얻은 두 관측값을 곱합니다.</strong>

기댓값을 나타내는 기호 $\langle P\rangle$는 현재 상태에서 관측량 $P$를 측정했을 때의 평균입니다. 예를 들어

$$
\langle Z_0Z_1\rangle
=
\langle\psi|Z_0Z_1|\psi\rangle
$$

입니다.

여기서 <strong>두 관측값의 곱을 평균하는 것</strong>과 <strong>각 관측값의 평균을 따로 구한 뒤 곱하는 것</strong>을 구분해야 합니다.

$$
\langle Z_0Z_1\rangle
\quad\text{와}\quad
\langle Z_0\rangle\langle Z_1\rangle
$$

은 일반적으로 다릅니다. 공동 관측량의 기댓값에는 두 측정 결과 사이의 상관관계가 반영됩니다. 따라서 $Z_0Z_1$ 항을 계산할 때에는 같은 시행에서 얻은 결과끼리 곱한 값을 사용합니다.

이제 Hamiltonian의 각 항에 기댓값을 취하면 에너지는

$$
E
=
-\langle Z_0Z_1\rangle
-\langle X_0Y_1\rangle
-\frac12\langle Z_0\rangle
-\frac12\langle Z_1\rangle
$$

가 됩니다.

각 기댓값을 서로 다른 실험에서 구하더라도, 모두 <strong>같은 상태를 준비했을 때의 값</strong>이어야 합니다. 서로 다른 각도로 준비한 상태들의 기댓값을 섞으면, 한 상태의 에너지를 계산한 것이 아닙니다.

---

### 함께 계산하기: $\lvert00\rangle$의 에너지

처음부터 네 항을 한꺼번에 다루기보다, 간단한 상태에서 부호와 계수를 확인해 보겠습니다.

상태 $|00\rangle$에서 두 큐비트를 Z 측정하면 모두 비트 0이 나옵니다. Z 측정의 비트 0은 관측값 $+1$에 대응하므로

$$
\langle Z_0\rangle=1,\qquad
\langle Z_1\rangle=1,\qquad
\langle Z_0Z_1\rangle=1
$$

입니다.

한편 $|0\rangle$을 X나 Y 방향으로 측정하면 관측값 $+1$과 $-1$이 같은 확률로 나옵니다. $|00\rangle$은 두 큐비트의 곱상태이므로, q0의 X 측정과 q1의 Y 측정 결과도 독립적입니다. 따라서 두 관측값의 곱은 $+1$과 $-1$이 같은 확률로 나와

$$
\langle X_0Y_1\rangle=0
$$

입니다.

이를 에너지 식에 넣으면

$$
E_{|00\rangle}
=
-1-0-\frac12-\frac12
=
-2
$$

를 얻습니다.

이 값은 앞으로 작성할 에너지 함수의 첫 확인 기준입니다. 함수는 $|00\rangle$뿐 아니라 다른 상태에서 얻은 기댓값도 같은 규칙으로 처리해야 합니다.

---

### 측정 비트를 관측값으로 읽기

Qiskit의 측정 결과에는 $+1,-1$ 대신 비트 $0,1$이 기록됩니다. Pauli 관측량의 기댓값을 계산하려면 이 비트를 다음과 같이 해석합니다.

$$
b=0\;\longleftrightarrow\;+1,
\qquad
b=1\;\longleftrightarrow\;-1.
$$

이를 한 식으로 쓰면 관측값은 $(-1)^b$입니다.

q0와 q1의 결과를 각각 $b_0,b_1$이라고 하면, 한 번의 Z 설정 실험에서 얻는 값들은

$$
z_0=(-1)^{b_0},\qquad
z_1=(-1)^{b_1},\qquad
z_0z_1=(-1)^{b_0+b_1}
$$

입니다.

같은 설정을 $N$번 실행했다면, 각 시행에서 얻은 값을 평균하여 기댓값을 추정합니다. 예를 들어

$$
\widehat{\langle Z_0Z_1\rangle}
=
\frac1N\sum_{j=1}^{N}z_0^{(j)}z_1^{(j)}
$$

입니다. 위의 $\widehat{\phantom{E}}$ 표시는 유한한 측정 기록으로 구한 <strong>추정값</strong>이라는 뜻입니다.

이 식에서 $z_0^{(j)}$와 $z_1^{(j)}$는 같은 $j$번째 시행의 결과입니다. 각 큐비트의 결과를 따로 섞거나, 서로 다른 시행의 결과를 임의로 짝짓지 않습니다.

이번 Lab의 앞부분에서는 제공 도구로 정확한 기댓값을 계산합니다. 뒤에서 유한 shots를 사용할 때에도, 측정 비트를 관측값으로 읽고 평균하는 원리는 같습니다.

---

### X와 Y 측정도 Z 측정 장치로 읽을 수 있습니다

회로 끝에서 Z 측정을 하더라도, 그 전에 기저를 바꾸면 원래 상태의 X 또는 Y 관측량을 읽을 수 있습니다.

X 측정부터 살펴보겠습니다. X의 두 고유상태는

$$
|+\rangle=\frac{|0\rangle+|1\rangle}{\sqrt2},
\qquad
|-\rangle=\frac{|0\rangle-|1\rangle}{\sqrt2}
$$

이고, 각각 관측값 $+1,-1$에 대응합니다. Hadamard 게이트는

$$
H|+\rangle=|0\rangle,
\qquad
H|-\rangle=|1\rangle
$$

로 작용합니다. 따라서 <strong>H를 적용한 뒤 Z 측정</strong>하면, 원래 상태를 X 측정한 결과를 비트 0과 1로 읽을 수 있습니다.

Y 측정도 같은 생각으로 구현합니다. Y의 두 고유상태는

$$
|+i\rangle=\frac{|0\rangle+i|1\rangle}{\sqrt2},
\qquad
|-i\rangle=\frac{|0\rangle-i|1\rangle}{\sqrt2}
$$

입니다. 각각 Y 관측값 $+1,-1$에 대응합니다.

위상 게이트의 역연산 $S^\dagger$는

$$
S^\dagger|0\rangle=|0\rangle,
\qquad
S^\dagger|1\rangle=-i|1\rangle
$$

로 작용하므로

$$
|+i\rangle
\xrightarrow{S^\dagger}
|+\rangle
\xrightarrow{H}
|0\rangle,
$$

$$
|-i\rangle
\xrightarrow{S^\dagger}
|-\rangle
\xrightarrow{H}
|1\rangle
$$

가 됩니다. 따라서 Y 측정에는 <strong>$S^\dagger$를 먼저 적용하고, 그다음 H를 적용한 뒤 Z 측정</strong>합니다.

이 순서에서 기저 변환은 원래 상태의 관측량을 읽기 위한 준비입니다. q0에 H를 적용한 뒤 얻은 비트는 원래 상태의 X 관측값으로, q1에 $S^\dagger$와 H를 적용한 뒤 얻은 비트는 원래 상태의 Y 관측값으로 해석합니다.

이번 실습에서는 다음 두 측정 설정을 사용합니다.

| 측정 설정 | q0에 적용할 측정 전 연산 | q1에 적용할 측정 전 연산 | 기록에서 구할 기댓값 |
|---|---|---|---|
| <strong>Z 설정</strong> | 없음 | 없음 | $\langle Z_0\rangle,\ \langle Z_1\rangle,\ \langle Z_0Z_1\rangle$ |
| <strong>XY 설정</strong> | H | $S^\dagger$ 다음 H | $\langle X_0Y_1\rangle$ |

Z 설정에서는 한 시행의 두 결과로 $z_0,z_1,z_0z_1$을 모두 구할 수 있습니다. 따라서 같은 측정 기록을 이용해 세 기댓값을 각각 계산합니다.

XY 설정에서는 각 시행의 X 관측값과 Y 관측값을 곱한 뒤 평균합니다. 이 설정을 실행할 때에는 <strong>Z 설정에 사용했던 것과 같은 상태를 새로 준비</strong>해야 합니다. 앞에서 Z 측정을 마친 데이터에 기저 변환을 이어 붙이면, 원래 상태가 아니라 Z 측정 이후의 상태를 조사하게 됩니다.

---

### 수식의 큐비트 번호와 Qiskit 문자열 맞추기

수식에서는 아래첨자로 연산이 작용하는 큐비트를 표시했습니다. Qiskit의 Pauli 문자열에서는 각 글자의 위치가 큐비트를 정합니다.

두 큐비트의 Pauli 문자열은 <strong>왼쪽이 q1, 오른쪽이 q0</strong>입니다. 따라서 대응은 다음과 같습니다.

| 수학 표기 | Qiskit Pauli 문자열 | 각 큐비트에 작용하는 연산 |
|---|---|---|
| $Z_0Z_1$ | `"ZZ"` | q1에 Z, q0에 Z |
| $X_0Y_1$ | `"YX"` | q1에 Y, q0에 X |
| $Z_0$ | `"IZ"` | q1에 항등 연산, q0에 Z |
| $Z_1$ | `"ZI"` | q1에 Z, q0에 항등 연산 |

특히 수식에 $X_0Y_1$이라고 적혀 있다는 이유로 문자열을 `"XY"`로 쓰지 않도록 주의하세요. `"XY"`는 q1에 X, q0에 Y가 작용하는 다른 관측량입니다.

상태벡터의 기저도

$$
|q_1q_0\rangle
$$

순서입니다. 예를 들어 $|01\rangle$은 q1이 $|0\rangle$, q0가 $|1\rangle$인 상태입니다.

아래 문제에서는 이 순서를 일관되게 사용합니다.

---

### 직접 풀어보기

### (a) 서로 다른 상태의 에너지 계산하기

다음 두 상태를 살펴보세요.

$$
|\psi_A\rangle
=
|0\rangle_{q_1}|1\rangle_{q_0}
=
|01\rangle,
$$

$$
|\psi_B\rangle
=
|+i\rangle_{q_1}|+\rangle_{q_0}.
$$

각 상태에서 네 관측량의 기댓값과 에너지를 구하여 표를 완성하세요.

| 상태 | $\langle Z_0Z_1\rangle$ | $\langle X_0Y_1\rangle$ | $\langle Z_0\rangle$ | $\langle Z_1\rangle$ | 에너지 $E$ |
|---|---:|---:|---:|---:|---:|
| $\lvert\psi_A\rangle$ | … | … | … | … | … |
| $\lvert\psi_B\rangle$ | … | … | … | … | … |

표를 완성한 뒤 다음을 설명하세요.

- $|\psi_A\rangle$에서 q0와 q1의 상태를 반대로 읽으면, 표의 어느 기댓값이 달라지나요? 최종 에너지 하나만 비교해도 이 실수를 반드시 발견할 수 있을까요?
- $|\psi_B\rangle$의 $X_0Y_1$ 기댓값을 구하려고 Qiskit 문자열 `"XY"`를 사용하면 어떤 관측량을 계산하게 되나요? 이 상태에서 올바른 관측량과 같은 결과를 주는지 확인하세요.

계산 후에는 제공 도구의 결과와 비교합니다. 에너지만 맞는지 확인하지 말고, <strong>각 기댓값이 의도한 큐비트와 관측량에 대응하는지</strong>도 함께 확인하세요.

---

### 1a 답안

실행 전 예상과 계산:

…

실행 결과와 설명:

…

### 결과 확인 실습 1a

수작업 계산을 먼저 적은 뒤 실행하세요. 이 실습에서는 제공된 물리 계산 도구로 결과를 확인합니다. 아래 결과가 대기로 표시되면 앞의 실습 코드 실행 설정과 확인 셀을 점검하세요.

In [ ]:
def experiment_1a():
    first = QuantumCircuit(2)
    first.x(0)
    second = QuantumCircuit(2)
    second.h(0)
    second.h(1)
    second.s(1)
    results, rows = [], []
    for label, circuit in (("A: |01>", first), ("B: |+i> on q1, |+> on q0", second)):
        values = expectations_from_circuit(circuit)
        energy = exact_energy(circuit)
        state = circuit_state(circuit)
        other_xy = float(np.real(state.expectation_value(SparsePauliOp("XY"))))
        results.append({"label": label, "values": values, "energy": energy, "wrong_xy": other_xy})
        rows.append([label, *[values[k] for k in OBSERVABLE_KEYS], energy, other_xy])
    show_table(["상태", "ZZ", "YX", "IZ", "ZI", "정확한 E", "비교: XY"], rows)
    return results

if RUN_OBSERVATIONS:
    _ = run_demo("1a", experiment_1a)
else:
    LAB_RESULTS.pop("1a", None)
    print("[대기] 1(a): 예상과 계산을 적은 뒤 RUN_OBSERVATIONS를 True로 바꾸세요.")

### (b) 기댓값을 에너지로 바꾸는 함수 작성하기

다음 함수를 작성하세요.

```python
energy_from_expectations(values, xy_coeff=-1.0)
```

`values`는 네 관측량의 기댓값을 담은 딕셔너리입니다.

| 키 | 저장된 값 |
|---|---|
| `"ZZ"` | $\langle Z_0Z_1\rangle$ |
| `"YX"` | $\langle X_0Y_1\rangle$ |
| `"IZ"` | $\langle Z_0\rangle$ |
| `"ZI"` | $\langle Z_1\rangle$ |

`xy_coeff`는 $X_0Y_1$ 항의 계수입니다. 기본값 $-1.0$은 현재 문제의 Hamiltonian에 해당합니다. 다른 값 $c$가 전달되면

$$
\mathcal H(c)
=
-Z_0Z_1+cX_0Y_1-\frac12(Z_0+Z_1)
$$

의 에너지를 계산합니다. 나머지 세 항의 계수는 유지됩니다.

함수는 <strong>입력받은 기댓값과 계수로 계산한 에너지 하나를 실수로 반환</strong>해야 합니다. 상태 준비, 측정, 최적화는 이 함수에서 수행하지 않습니다.

먼저 앞의 $|00\rangle$ 예제로 부호와 계수를 확인하세요. 이어서 (a)의 두 상태와 제공된 추가 상태에서도 함수가 올바르게 작동하는지 검사합니다. 특정 상태의 에너지를 직접 반환하지 말고, 전달된 네 값을 사용하세요.

확인 셀은 여러 상태와 계수에서 반환값을 독립적인 Hamiltonian 기댓값 계산과 비교합니다. 이 함수를 완성하면 뒤의 회로 실험과 최적화에서도 같은 평가기를 사용합니다.

---

In [ ]:
def energy_from_expectations(values, xy_coeff=-1.0):
    # TODO BEGIN — 전달된 네 기댓값과 계수로 에너지를 계산하세요.
    raise NotImplementedError('energy_from_expectations')
    # TODO END

In [ ]:
_ = check_energy_from_expectations(energy_from_expectations)

### 1b 답안

계산·실행 결과와 근거:

…

### (c) 에너지 평가에 필요한 측정 실험 설계하기

한 각도로 준비한 상태의 에너지를 측정하려고 합니다. Z 설정을 $N$회, XY 설정을 $N$회 실행하며, 각 시행에서는 상태를 새로 준비하고 두 큐비트를 측정합니다.

다음 표를 완성하세요.

| 구하려는 기댓값 | 사용할 측정 설정 | 한 시행에서 사용할 값 | 평균에 사용하는 시행 수 |
|---|---|---|---:|
| $\langle Z_0\rangle$ | … | … | … |
| $\langle Z_1\rangle$ | … | … | … |
| $\langle Z_0Z_1\rangle$ | … | … | … |
| $\langle X_0Y_1\rangle$ | … | … | … |

이어서 다음 세 설명을 검토하세요. 각 설명이 적절한지 판단하고, 잘못된 경우에는 올바른 계산이나 실험 절차로 고쳐 쓰세요.

1. <strong>“기댓값이 네 개이므로, 에너지 한 번을 평가하려면 상태를 총 $4N$번 준비해야 한다.”</strong>  
   어느 기댓값들을 같은 측정 기록에서 얻을 수 있는지 근거로 설명하세요.

2. <strong>“상태 준비 횟수를 줄이기 위해, Z 측정을 마친 두 큐비트에 H와 $S^\dagger$를 적용하여 XY 설정도 이어서 측정하겠다.”</strong>  
   두 번째 설정이 어떤 상태의 관측량을 읽게 되는지 설명하세요. 원래 준비한 상태의 에너지를 구하려면 실험을 어떻게 구성해야 하나요?

3. <strong>“$Z_0Z_1$의 기댓값은 q0의 Z 평균과 q1의 Z 평균을 곱해서 구하면 된다.”</strong>  
   같은 시행의 두 결과를 먼저 곱한 뒤 평균하는 방법과 비교하세요. 두 방법이 일반적으로 같다고 볼 수 있는지, 앞의 Lab에서 다룬 상태를 예로 들어 설명하세요.

마지막으로, 완성한 에너지 평가기에 넣을 네 값이 <strong>같은 상태를 대상으로 계산되었는지 확인하려면 무엇을 기록해야 하는지</strong> 한두 문장으로 정리하세요.

### 1c 답안

계산·실행 결과와 근거:

…

## 문제 2. 각도를 바꾸면 바닥상태에 도달할 수 있을까요?

앞 문제에서는 상태의 네 기댓값을 에너지 하나로 합치는 평가기를 만들었습니다. 이제 이 평가기를 사용해 <strong>회로의 각도를 바꾸었을 때 에너지가 어떻게 달라지는지</strong> 살펴보겠습니다.

각도를 충분히 많이 시험하면 어떤 상태든 준비할 수 있을까요? 각도를 바꾸어 만들 수 있는 상태는 회로 구조에 따라 정해집니다. 따라서 에너지 탐색을 시작하기 전에, 이번 회로가 어떤 상태들을 준비하는지 확인할 필요가 있습니다.

이번 문제에서는 복습자료에서 사용했던 회로로 시작합니다. 두 큐비트를 $|00\rangle$에 준비하고 다음 순서로 게이트를 적용합니다.

1. q0에 $R_y(\theta)$를 적용합니다.
2. q0를 제어 큐비트, q1을 표적 큐비트로 하는 CX를 적용합니다.

이 회로에서 바꿀 수 있는 매개변수는 각도 $\theta$ 하나입니다. 회로 코드는 제공되며, 여러분은 <strong>회로가 만드는 상태와 그 상태의 에너지</strong>를 계산합니다. 그 결과를 바탕으로, 각도 탐색만으로 바닥 에너지에 도달할 수 있는지 판단하겠습니다.

---

### 회전 게이트는 중첩의 계수를 바꿉니다

먼저 $R_y(\theta)$가 어떤 상태를 만드는지 확인하겠습니다. 이 게이트의 행렬은

$$
R_y(\theta)
=
\begin{pmatrix}
\cos(\theta/2)&-\sin(\theta/2)\\
\sin(\theta/2)&\cos(\theta/2)
\end{pmatrix}
$$

입니다. 따라서 $|0\rangle$에 적용하면

$$
R_y(\theta)|0\rangle
=
\cos\frac{\theta}{2}|0\rangle
+
\sin\frac{\theta}{2}|1\rangle
$$

이 됩니다.

각도 $\theta$를 바꾸면 두 기저 성분의 계수가 달라집니다. 두 계수를 각각 제곱한 값은 Z 측정에서 0과 1을 얻을 확률이며,

$$
\cos^2\frac{\theta}{2}
+
\sin^2\frac{\theta}{2}
=1
$$

이므로 상태는 정규화되어 있습니다.

이번 회로에서는 q0에만 이 회전을 적용합니다. 따라서 회전 직후 q1은 여전히 $|0\rangle$입니다. 두 큐비트의 공동 상태를 쓸 때에는 <strong>q1의 상태와 회전된 q0의 상태를 함께 적어야 합니다.</strong>

상태벡터의 기저 순서는 앞 문제와 같이 $|q_1q_0\rangle$입니다. q0의 성분이 달라지면 켓의 오른쪽 비트가 달라집니다.

---

### CX는 중첩의 각 성분에 따로 작용합니다

다음으로 CX를 적용합니다. 이번 CX의 제어는 q0, 표적은 q1입니다. 따라서 q0가 1인 성분에서 q1을 뒤집습니다.

$|q_1q_0\rangle$ 순서로 쓰면 기저 상태에 대한 작용은 다음과 같습니다.

| CX 적용 전 | CX 적용 후 |
|---|---|
| $\lvert00\rangle$ | $\lvert00\rangle$ |
| $\lvert01\rangle$ | $\lvert11\rangle$ |
| $\lvert10\rangle$ | $\lvert10\rangle$ |
| $\lvert11\rangle$ | $\lvert01\rangle$ |

입력이 중첩 상태라면 각 성분에 이 규칙을 적용하고, 원래 계수를 그대로 붙입니다. 예를 들어 서로 다른 두 기저 상태 $|u\rangle,|v\rangle$에 대해

$$
\mathrm{CX}
\bigl(a|u\rangle+b|v\rangle\bigr)
=
a\,\mathrm{CX}|u\rangle
+
b\,\mathrm{CX}|v\rangle
$$

입니다.

이 회로에서 $R_y$는 q0의 두 성분에 붙는 계수를 조절합니다. 이어지는 CX는 q0의 성분에 따라 q1의 상태를 바꾸어, 두 큐비트의 측정 결과 사이에 관계를 만듭니다. 어떤 공동 상태가 되는지는 위의 규칙을 적용해 직접 구해 보겠습니다.

CX가 회로에 들어 있다는 사실만으로 모든 각도에서 얽힘이 생기는 것은 아닙니다. 또한 얽힌 상태를 만들 수 있더라도, 모든 얽힌 상태를 만들 수 있다는 뜻은 아닙니다. 이번에는 <strong>이 특정한 회로와 매개변수로 준비할 수 있는 상태의 범위</strong>를 확인합니다.

---

### 계수의 크기와 상대위상을 구분하기

회로가 만드는 상태를 살펴볼 때에는 각 성분의 확률뿐 아니라 계수의 부호와 위상도 확인해야 합니다.

두 직교하는 기저 상태의 중첩을

$$
|\chi\rangle=a|u\rangle+b|v\rangle
$$

라고 하겠습니다. $|a|^2,|b|^2$는 해당 기저에서 두 성분을 얻을 확률입니다. 그러나 두 확률만으로 $a,b$를 모두 알 수는 없습니다. 예를 들어 계수 $b$에 위상 $e^{i\phi}$가 붙어도 $|b|^2$는 변하지 않습니다.

$$
a|u\rangle+b|v\rangle,
\qquad
a|u\rangle+e^{i\phi}b|v\rangle
$$

는 같은 기저에서 같은 측정 확률을 줄 수 있지만, 두 성분 사이의 상대위상은 다를 수 있습니다.

이번 $R_y$의 행렬 원소는 모두 실수이고, CX도 기저 상태를 서로 옮길 때 복소 위상을 추가하지 않습니다. 따라서 초기 상태 $|00\rangle$에서 출발한 이번 회로의 계산 기저 진폭은 실수입니다.

실수 계수도 양수와 음수가 될 수 있으므로, 각도에 따라 성분 사이의 부호가 달라질 수는 있습니다. 다만 이것이 임의의 복소 상대위상을 조절하는 것과 같은지는 구분해야 합니다.

<strong>이런 계수의 제한이 에너지에도 영향을 주는지</strong>는 Hamiltonian의 각 항을 계산해서 확인하겠습니다. 특히 $X_0Y_1$ 항에는 Y 관측량이 들어 있으므로, 계산 중 나타나는 복소 위상을 빠뜨리지 않아야 합니다.

---

### 상태에서 기댓값을 계산하는 방법

기댓값을 구하려면 먼저 관측량을 상태에 적용하고, 그 결과와 원래 상태의 내적을 계산합니다.

$$
\langle P\rangle
=
\langle\psi|\bigl(P|\psi\rangle\bigr).
$$

한 큐비트의 정규화된 상태

$$
|\chi\rangle=a|0\rangle+b|1\rangle
$$

를 예로 들어 보겠습니다. Z를 적용하면

$$
Z|\chi\rangle=a|0\rangle-b|1\rangle
$$

이고, 원래 상태의 브라는

$$
\langle\chi|
=
a^*\langle0|+b^*\langle1|
$$

입니다. 따라서 서로 다른 기저 상태의 내적이 0이라는 사실을 사용하면

$$
\langle Z\rangle
=
(a^*\langle0|+b^*\langle1|)
(a|0\rangle-b|1\rangle)
=
|a|^2-|b|^2
$$

를 얻습니다.

두 큐비트에서도 같은 방법을 사용합니다. 필요한 Pauli 연산의 작용은 다음과 같습니다.

| 연산 | $\lvert0\rangle$에 적용 | $\lvert1\rangle$에 적용 |
|---|---|---|
| $X$ | $\lvert1\rangle$ | $\lvert0\rangle$ |
| $Y$ | $i\lvert1\rangle$ | $-i\lvert0\rangle$ |
| $Z$ | $\lvert0\rangle$ | $-\lvert1\rangle$ |

예를 들어 $X_0Y_1$을 적용할 때에는 q0에 X, q1에 Y를 적용합니다. 각 성분에서 얻은 위상까지 계수에 곱한 다음, 원래 상태와 내적을 계산합니다.

Y가 포함되어 있으면 중간 계산에 $i$가 나타날 수 있습니다. 이를 임의로 지우거나 계수의 절댓값으로 바꾸지 마세요. Pauli 관측량의 최종 기댓값은 실수이지만, 그 값을 얻는 과정에는 복소 계수와 그 켤레가 함께 사용됩니다.

---

### 개별 각도의 실행 결과와 회로 전체의 한계를 구분하기

회로가 준비한 상태를 $|\psi_{\mathrm r}(\theta)\rangle$, 그 에너지를

$$
E_{\mathrm r}(\theta)
=
\langle\psi_{\mathrm r}(\theta)|
\mathcal H
|\psi_{\mathrm r}(\theta)\rangle
$$

라고 쓰겠습니다. 아래첨자 r은 이번 실수 진폭 회로를 구분하기 위한 표시입니다.

제공된 스캔 실험은 여러 각도에서 회로를 실행하고 다음 값을 표로 보여 줍니다.

- 네 Pauli 관측량의 정확한 기댓값
- 앞 문제에서 작성한 함수로 계산한 에너지
- 기준 바닥 에너지 $E_0$와의 차이

이 실험에는 유한 shots의 표본 변동이 없습니다. 따라서 표에 나타난 변화는 각도에 따라 준비되는 상태와 그 기댓값의 변화입니다.

다만 유한한 개수의 각도를 시험하는 것만으로 모든 각도에서의 최솟값이 증명되지는 않습니다. 스캔에서 가장 낮았던 점 사이에 더 낮은 값이 있을 수도 있기 때문입니다.

이번에는 회로가 단순하므로 에너지를 $\theta$의 식으로 직접 구할 수 있습니다. <strong>스캔은 수식의 예측을 확인하는 데 사용하고, 전체 각도 범위에서 가능한 최솟값은 수식으로 판단</strong>하겠습니다.

계산을 정리할 때에는 다음 관계를 사용할 수 있습니다.

$$
\cos^2\frac{\theta}{2}
-
\sin^2\frac{\theta}{2}
=
\cos\theta,
$$

$$
2\sin\frac{\theta}{2}\cos\frac{\theta}{2}
=
\sin\theta.
$$

이후 $\sin\theta,\cos\theta$가 가질 수 있는 값의 범위를 이용하면, 에너지가 어디까지 내려갈 수 있는지 판단할 수 있습니다.

---

### 직접 풀어보기

### (a) 회로가 준비하는 상태 구하기

초기 상태는

$$
|00\rangle=|0\rangle_{q_1}|0\rangle_{q_0}
$$

입니다. q0에 $R_y(\theta)$, 이어서 $\mathrm{CX}(0,1)$을 적용했을 때의 상태를 차례로 구하세요.

| 단계 | 두 큐비트의 공동 상태 |
|---|---|
| 초기 상태 | $\lvert00\rangle$ |
| q0에 $R_y(\theta)$ 적용 후 | … |
| $\mathrm{CX}(0,1)$ 적용 후 | … |

마지막 행의 상태를 $|\psi_{\mathrm r}(\theta)\rangle$라고 하겠습니다.

상태식을 구한 뒤 다음을 설명하세요.

- $|00\rangle,|01\rangle,|10\rangle,|11\rangle$ 중 어느 성분이 최종 상태에 나타날 수 있나요?
- $\theta$를 바꾸면 그 성분들의 계수는 어떻게 달라지나요?
- 이 회로가 준비하는 상태에서 두 비영 성분 사이에 임의의 복소 상대위상을 붙일 수 있나요? 사용한 게이트의 작용을 근거로 설명하세요.

제공된 상태 계산 도구로 몇 가지 각도의 결과를 확인하되, 답안에는 특정 각도의 벡터만 적지 말고 <strong>일반적인 $\theta$에 대한 상태식</strong>을 작성하세요.

---

### 제공된 회로와 상태 확인 도구

`real_ansatz(theta)`는 이 문제의 RY → CX 회로를 반환합니다. 상태벡터를 확인하려면 `circuit_state(real_ansatz(theta)).data`를 사용할 수 있습니다. 벡터 성분의 순서는 `00, 01, 10, 11`이며, 먼저 일반적인 각도의 상태식을 적은 뒤 특정 각도에서 대조하세요.

### 2a 답안

계산·실행 결과와 근거:

…

### (b) 각도에 따른 에너지 식 구하기

(a)에서 구한 상태를 사용하여 다음 네 기댓값을 $\theta$의 식으로 나타내세요.

| 관측량 | $\lvert\psi_{\mathrm r}(\theta)\rangle$에서의 기댓값 |
|---|---|
| $Z_0Z_1$ | … |
| $X_0Y_1$ | … |
| $Z_0$ | … |
| $Z_1$ | … |

$X_0Y_1$ 항은 다음 순서로 계산 과정을 남기세요.

1. $X_0Y_1$을 최종 상태의 각 기저 성분에 적용합니다.
2. 그 결과를 합하여 $X_0Y_1|\psi_{\mathrm r}(\theta)\rangle$를 구합니다.
3. 원래 상태의 브라와 내적을 계산합니다.

이어서 네 기댓값을 Hamiltonian의 계수에 맞게 합하여

$$
E_{\mathrm r}(\theta)
=
-\langle Z_0Z_1\rangle
-\langle X_0Y_1\rangle
-\frac12\langle Z_0\rangle
-\frac12\langle Z_1\rangle
$$

를 하나의 식으로 정리하세요.

각도를 바꾸었을 때 <strong>변하는 기댓값과 변하지 않는 기댓값</strong>도 구분해서 적습니다. 특히 $X_0Y_1$ 항의 결과가 특정 각도에서만 성립하는지, 이번 상태군의 모든 각도에서 성립하는지 자신의 계산을 근거로 설명하세요.

마지막으로 제공된 스캔 실험을 실행하여, 수식에서 예상한 네 기댓값과 에너지가 코드의 결과와 일치하는지 확인하세요.

---

### 2b 답안

실행 전 예상과 계산:

…

실행 결과와 설명:

…

### 결과 확인 실습 2bc

먼저 해당 문항의 계산과 예상을 적으세요. 필요한 코드 검사를 통과한 뒤 이 실험을 실행하고 실행 결과를 답안에 덧붙입니다. 이 표는 2(b)의 식 확인과 이어지는 2(c)의 최솟값 판단에 함께 사용합니다. 아래 결과가 대기로 표시되면 앞의 실습 코드 실행 설정과 확인 셀을 점검하세요.

In [ ]:
def experiment_2bc():
    rows, results = [], []
    for theta in np.linspace(0, 2*np.pi, 13):
        circuit = real_ansatz(theta)
        values = expectations_from_circuit(circuit)
        energy = energy_from_expectations(values)
        results.append({"theta": float(theta), "values": values, "energy": float(energy)})
        rows.append([theta, *[values[k] for k in OBSERVABLE_KEYS], energy])
    show_table(["theta", "ZZ", "YX", "IZ", "ZI", "학생 함수의 E"], rows)
    best = min(results, key=lambda row: row["energy"])
    show_table(["스캔에서 선택된 theta", "스캔 최저 E", "정확한 E0"],
               [[best["theta"], best["energy"], ground_reference()["energy"]]])
    print("이 표는 유한한 각도 스캔입니다. 모든 각도에서의 최솟값은 자신의 식으로 판단하세요.")
    return {"rows": results, "best": best}

if RUN_OBSERVATIONS:
    _ = run_demo("2bc", experiment_2bc, (CHECK_LABELS[0],))
else:
    LAB_RESULTS.pop("2bc", None)
    print("[대기] 2(b,c): 상태식과 에너지 식을 작성한 뒤 관찰 실험을 실행하세요.")

In [ ]:
if RUN_OBSERVATIONS and "2bc" in LAB_RESULTS:
    reference_energy = ground_reference()["energy"]
    show_table(["theta", "E-E0"],
               [[row["theta"], row["energy"] - reference_energy]
                for row in LAB_RESULTS["2bc"]["rows"]])
else:
    print("[대기] 에너지 차이 표는 앞의 각도 스캔이 실행된 뒤 표시됩니다.")

### (c) 이 회로로 도달할 수 있는 최솟값 판단하기

(b)에서 구한 에너지 식을 사용하여 다음 값을 구하세요.

$$
E_{\mathrm r,\min}
=
\min_\theta E_{\mathrm r}(\theta).
$$

최솟값뿐 아니라, 그 값이 최소라고 판단한 이유와 이를 달성하는 각도 하나를 함께 적으세요. 스캔에서 가장 낮게 나온 값을 그대로 옮기는 것으로 끝내지 말고, 에너지 식에 들어 있는 삼각함수의 범위를 이용해 설명합니다.

이어서 다음 표를 완성하세요.

| 비교 항목 | 결과와 근거 |
|---|---|
| 각도 스캔에서 얻은 가장 낮은 에너지 | … |
| 수식으로 구한 $E_{\mathrm r,\min}$ | … |
| 기준 바닥 에너지 $E_0$ | $-1-\sqrt2$ |
| 두 최솟값의 차이 $E_{\mathrm r,\min}-E_0$ | … |
| 현재 회로로 바닥 에너지에 도달할 수 있는가? | … |

이 결과를 바탕으로 다음 두 의견을 검토하세요.

> <strong>의견 A:</strong> “아직 좋은 각도를 찾지 못했을 뿐이므로, 같은 회로에서 더 많은 각도를 시험하면 바닥 에너지에 도달할 것이다.”

> <strong>의견 B:</strong> “이 회로에는 CX가 있고 얽힌 상태도 만들 수 있으므로, 바닥상태를 표현하기에 충분하다.”

각 의견이 이번 문제에서 타당한지 판단하고, <strong>회로의 상태식과 에너지 식</strong>을 근거로 설명하세요. 특정 최적화 프로그램의 성능이나 실행 시간만을 근거로 삼지 않습니다.

마지막으로, 이번 결과에서 회로를 수정할 필요가 있다고 판단했다면 <strong>현재 각도 $\theta$로 조절할 수 없는 상태의 성질이 무엇인지</strong> 적으세요. 다음 문제에서는 그 성질을 조절할 수 있도록 회로를 확장하겠습니다.

### 2c 답안

계산·실행 결과와 근거:

…

## 문제 3. 필요한 상태를 만들도록 회로를 바꿔 볼까요?

앞 문제에서는 각도 $\theta$를 바꾸어 첫 회로가 어떤 상태를 준비하는지 살펴보았습니다. 이어서 에너지 식을 구하고, 그 회로가 도달할 수 있는 최솟값을 바닥 에너지 $E_0$와 비교했습니다.

이번에는 <strong>첫 회로에서 조절하지 못했던 상대위상을 새로운 매개변수로 조절</strong>하겠습니다. 회로를 수정한 뒤에는 앞에서 작성한 에너지 평가기에 연결하고, 고전 최적화기로 두 각도를 함께 탐색합니다.

회로에 게이트를 추가하는 것만으로 작업이 끝나지는 않습니다. 추가한 각도가 실제로 원하는 상대위상을 바꾸는지, 최적화기가 여러분이 작성한 회로와 에너지 함수를 사용하고 있는지도 확인하겠습니다.

---

### 두 성분의 크기와 상대위상을 각각 조절하기

앞 문제의 회로는

$$
|\psi_{\mathrm r}(\theta)\rangle
=
\cos\frac{\theta}{2}|00\rangle
+
\sin\frac{\theta}{2}|11\rangle
$$

를 준비했습니다. $\theta$를 바꾸면 두 성분의 계수는 달라지지만, 계산 기저의 진폭은 실수로 남습니다.

이번에는 다음 상태를 준비하는 회로를 만들겠습니다.

$$
|\psi(\theta,\phi)\rangle
=
\cos\frac{\theta}{2}|00\rangle
+
e^{i\phi}\sin\frac{\theta}{2}|11\rangle.
$$

새로 추가한 각도 $\phi$는 $|11\rangle$ 성분에 붙는 위상을 조절합니다. $\phi=0$으로 두면 앞 문제의 상태식으로 돌아갑니다.

두 매개변수의 역할을 정리하면 다음과 같습니다.

| 매개변수 | 상태에서 조절하는 부분 |
|---|---|
| $\theta$ | $\lvert00\rangle$과 $\lvert11\rangle$ 성분의 계수 크기 |
| $\phi$ | 두 성분 사이에 추가되는 상대위상 |

이 상태를 Z 측정하면

$$
P(00)=\cos^2\frac{\theta}{2},
\qquad
P(11)=\sin^2\frac{\theta}{2}
$$

이고, 나머지 두 결과의 확률은 0입니다. $|e^{i\phi}|=1$이므로 이 확률에는 $\phi$가 나타나지 않습니다.

따라서 $\theta$를 고정하고 $\phi$만 바꾸면 <strong>Z 측정 확률은 같아도 공동 상태는 달라질 수 있습니다.</strong> 이번에는 Hamiltonian에 들어 있는 $X_0Y_1$ 기댓값과 에너지를 함께 살펴보며 그 변화를 확인하겠습니다.

두 성분 가운데 하나의 계수가 0이면, 그 상태에서 두 성분 사이의 상대위상을 확인할 수는 없습니다. $\phi$의 작용을 확인하는 실험에서는 두 성분이 모두 존재하는 $\theta$를 사용하겠습니다.

---

### $R_z$는 두 기저 성분에 서로 다른 위상을 붙입니다

상대위상을 조절할 도구로 $R_z(\phi)$를 사용하겠습니다. 이 게이트는

$$
R_z(\phi)
=
\begin{pmatrix}
e^{-i\phi/2}&0\\
0&e^{i\phi/2}
\end{pmatrix}
$$

이며, 기저 상태에 다음과 같이 작용합니다.

$$
R_z(\phi)|0\rangle=e^{-i\phi/2}|0\rangle,
\qquad
R_z(\phi)|1\rangle=e^{i\phi/2}|1\rangle.
$$

이를 한 큐비트의 중첩 상태에 적용하면

$$
R_z(\phi)
\bigl(a|0\rangle+b|1\rangle\bigr)
=
ae^{-i\phi/2}|0\rangle
+
be^{i\phi/2}|1\rangle
$$

입니다. 두 항에서 공통 인자 $e^{-i\phi/2}$를 꺼내면

$$
=
e^{-i\phi/2}
\bigl(a|0\rangle+e^{i\phi}b|1\rangle\bigr)
$$

로 쓸 수 있습니다.

이 식에는 서로 다른 두 종류의 위상이 나타납니다.

- 괄호 바깥의 $e^{-i\phi/2}$는 상태 전체에 곱해지는 <strong>전역위상</strong>입니다.
- 괄호 안의 $e^{i\phi}$는 두 성분 사이의 <strong>상대위상</strong>을 바꿉니다.

상태 전체에 같은 위상이 곱해져도 측정 확률은 달라지지 않습니다. 반면 두 성분 사이의 상대위상은 다른 기저의 측정 결과에 영향을 줄 수 있습니다.

따라서 회로가 목표 상태와 비교해 전역위상만 다르다면 같은 상태를 준비한 것으로 인정합니다. 하지만 두 성분 사이의 상대위상이 다르면, Z 측정 확률이 같더라도 목표 상태를 준비했다고 할 수 없습니다.

---

### 게이트를 적용하는 위치도 확인해야 합니다

$R_z$가 위상을 조절하는 게이트라는 사실만으로, 회로의 어느 위치에 넣어도 같은 효과가 생기지는 않습니다.

회로를 따라가며 다음 두 가지를 확인하세요.

1. $R_z$가 적용되는 시점에 어떤 기저 성분들이 존재하나요?
2. 그 성분들에 붙는 위상이 서로 다른가요, 아니면 상태 전체에 같은 위상이 곱해지나요?

이번 회로에서는 $R_y$가 중첩을 만들고, CX가 두 큐비트의 성분을 연결합니다. $R_z$를 그 과정의 어느 위치와 어느 큐비트에 적용하는지에 따라 최종 상태에 남는 효과가 달라질 수 있습니다.

회로를 설계할 때에는 게이트 이름을 나열하는 데서 그치지 말고, <strong>각 단계에서 상태가 어떻게 변하는지</strong> 확인하세요. 서로 다른 게이트 배치가 같은 최종 상태를 준비할 수도 있으므로, 검사는 특정 배치 하나보다 준비된 상태를 기준으로 합니다.

---

### 회로와 에너지 평가기를 목적함수로 연결하기

회로가 두 각도를 받아 상태를 준비할 수 있게 되면, 고전 최적화기에 전달할 목적함수를 만들 수 있습니다.

최적화기는 한 번에 다음과 같은 매개변수 묶음을 제안합니다.

$$
\boldsymbol\theta=(\theta,\phi).
$$

목적함수는 이 두 값을 받아 다음 과정을 수행합니다.

| 순서 | 수행할 일 |
|---|---|
| 1 | 입력에서 $\theta,\phi$를 읽습니다. |
| 2 | 두 각도로 상태 준비 회로를 만듭니다. |
| 3 | 그 회로가 준비한 상태의 네 Pauli 기댓값을 구합니다. |
| 4 | 문제 1의 에너지 평가기로 기댓값들을 합합니다. |
| 5 | 계산한 에너지 하나를 반환합니다. |

이번 문제에서 사용할 제공 도구는 다음과 같습니다.

```python
expectations_from_circuit(circuit)
```

이 도구는 입력 회로가 $|00\rangle$에서 준비한 상태를 계산하고, 다음 네 키를 가진 딕셔너리를 반환합니다.

```python
"ZZ", "YX", "IZ", "ZI"
```

각 키의 의미는 문제 1과 같습니다. 반환되는 값은 이상적인 상태의 <strong>정확한 기댓값</strong>이며, 유한 shots의 표본 변동은 포함하지 않습니다.

여러분은 이 도구를 자신이 작성한 두 함수 사이에 연결합니다.

$$
(\theta,\phi)
\;\longrightarrow\;
\text{상태 준비 회로}
\;\longrightarrow\;
\text{네 기댓값}
\;\longrightarrow\;
E.
$$

목적함수가 에너지를 반환하면, 최적화기는 그 값을 이용해 다음에 시도할 각도를 선택합니다. 최적화기 자체를 새로 구현할 필요는 없습니다.

---

### 최적화 기록은 여러 후보를 평가한 기록입니다

제공된 최적화 실험은 초기값

$$
(\theta,\phi)=(0.4,0.6)
$$

에서 시작합니다. 각도 단위는 라디안입니다.

실험 기록에는 목적함수에 전달된 각도와 그때 반환된 에너지가 저장됩니다. 이를 읽으면 최적화기가 어떤 상태들을 시험했는지 확인할 수 있습니다.

기록된 에너지가 매번 이전 값보다 낮아질 필요는 없습니다. 최적화기는 다음 방향을 정하기 위해 주변의 여러 후보를 평가할 수 있고, 그중에는 에너지가 더 높은 후보도 있을 수 있습니다. 또한 한 번의 최적화 단계에서 목적함수를 여러 번 호출할 수 있으므로, <strong>목적함수 평가 횟수와 최적화 반복 횟수는 같은 수라고 가정하지 않습니다.</strong>

이번 문제에서는 찾은 각도와 에너지를 기록하고, 수정한 회로의 동작을 확인합니다. 종료 메시지와 실제 바닥상태 도달 여부를 함께 검토하는 일은 다음 문제에서 이어서 다루겠습니다.

---

### 직접 풀어보기

### (a) 두 각도로 목표 상태를 준비하는 회로 작성하기

다음 함수를 작성하세요.

```python
phase_ansatz(theta, phi)
```

함수는 $|00\rangle$에서 시작하여 다음 상태를 준비하는 회로를 반환해야 합니다.

$$
|\psi(\theta,\phi)\rangle
=
\cos\frac{\theta}{2}|00\rangle
+
e^{i\phi}\sin\frac{\theta}{2}|11\rangle.
$$

함수의 조건은 다음과 같습니다.

- 입력 `theta`, `phi`는 라디안 단위의 실수 각도입니다.
- 반환값은 큐비트 두 개, 고전 비트가 없는 `QuantumCircuit(2)`입니다.
- $R_y$, $R_z$, CX 게이트를 사용합니다.
- 상태 준비를 직접 지정하는 `initialize`와 측정은 사용하지 않습니다.
- 특정 각도에서만 맞는 회로가 아니라, 전달받은 일반적인 $\theta,\phi$에서 목표 상태를 준비해야 합니다.
- 목표 상태 전체에 곱해지는 전역위상 차이는 허용합니다.

앞 문제의 회로를 출발점으로 삼고, $R_z$를 추가하여 상대위상을 조절하세요.

회로를 작성한 뒤에는 다음을 설명하세요.

1. $R_z$를 어느 큐비트의 어느 위치에 적용했나요?
2. 그 위치에서 $R_z$를 적용했을 때, 최종 두 성분에 각각 어떤 위상이 붙나요?
3. 상태 전체의 공통 위상을 꺼내면 목표 상태식과 어떻게 대응하나요?

확인 셀은 여러 각도에서 준비된 공동 상태를 목표 상태와 비교합니다. <strong>Z 측정 확률만 맞추는 것으로는 충분하지 않습니다.</strong>

---

In [ ]:
def phase_ansatz(theta, phi):
    circuit = QuantumCircuit(2)
    # TODO BEGIN — 두 각도에 해당하는 목표 공동 상태를 준비하세요.
    raise NotImplementedError('phase_ansatz')
    # TODO END
    return circuit

In [ ]:
_ = check_phase_ansatz(phase_ansatz)

### 3a 답안

계산·실행 결과와 근거:

…

### (b) 자신의 회로를 고전 최적화기에 연결하기

다음 함수를 작성하세요.

```python
vqe_objective(parameters)
```

`parameters`는 길이가 2인 매개변수 묶음이며, 순서는 다음과 같습니다.

```python
theta, phi
```

목적함수에서는 다음 도구들을 연결합니다.

| 도구 | 역할 |
|---|---|
| 자신이 작성한 `phase_ansatz` | 두 각도에 해당하는 회로 준비 |
| 제공된 `expectations_from_circuit` | 회로 상태의 네 기댓값 계산 |
| 자신이 작성한 `energy_from_expectations` | 기댓값들을 에너지 하나로 변환 |

현재 Hamiltonian의 $X_0Y_1$ 계수는 $-1$입니다. 목적함수는 이 Hamiltonian의 에너지 하나를 실수로 반환해야 합니다.

정확한 바닥 에너지나 이미 알고 있는 최적 각도를 함수 안에 넣지 마세요. 최적화기는 여러 후보 각도에서 이 함수를 호출하므로, <strong>매번 입력받은 각도로 준비한 상태의 에너지</strong>를 계산해야 합니다.

확인 셀을 통과한 뒤 제공된 최적화 실험을 실행하고, 다음을 기록하세요.

| 항목 | 결과 |
|---|---|
| 사용한 초기값 | $(0.4,0.6)$ |
| 최적화기가 반환한 $\theta,\phi$ | … |
| 반환된 각도에서의 에너지 | … |
| 목적함수 평가 횟수 | … |
| 최적화기의 종료 메시지 | … |

이어서 반환된 각도를 `vqe_objective`에 다시 넣어, 보고된 에너지가 재현되는지 확인하세요.

마지막으로 첫 회로의 최소 에너지와 이번 결과를 비교하고, <strong>회로를 확장한 뒤 에너지가 어떻게 달라졌는지</strong> 적으세요. 수치 비교와 함께, 추가한 매개변수가 상태의 어떤 성질을 조절하도록 했는지도 설명합니다.

---

In [ ]:
def vqe_objective(parameters):
    # TODO BEGIN — 회로 준비, 기댓값 계산, 에너지 평가를 연결하세요.
    raise NotImplementedError('vqe_objective')
    # TODO END

In [ ]:
_ = check_vqe_objective(vqe_objective)

### 3b 답안

계산·실행 결과와 근거:

…

### 결과 확인 실습 3b

먼저 해당 문항의 계산과 예상을 적으세요. 필요한 코드 검사를 통과한 뒤 이 실험을 실행하고 실행 결과를 답안에 덧붙입니다. 아래 결과가 대기로 표시되면 앞의 실습 코드 실행 설정과 확인 셀을 점검하세요.

In [ ]:
def experiment_3b():
    report = optimize_vqe(vqe_objective, initial=(0.4, 0.6))
    recalculated = float(vqe_objective(report["parameters"]))
    assert np.isclose(recalculated, report["objective_energy"], atol=ATOL, rtol=0), "반환된 각도에서의 에너지를 다시 확인하세요."
    report["recalculated_energy"] = recalculated
    show_table(["초기값", "반환 theta,phi", "목적함수 E", "재계산 E", "평가 횟수", "반복 횟수"],
               [[report["initial"], report["parameters"], report["objective_energy"], recalculated, report["nfev"], report["nit"]]])
    print("최적화기 종료:", report["success"], report["message"])
    show_table(["평가 순서", "theta,phi", "에너지"],
               [[index+1, row["parameters"], row["energy"]] for index, row in enumerate(report["history"])])
    print("위 표는 목적함수 평가 순서입니다. 평가 횟수와 최적화 반복 횟수는 구분합니다.")
    return report

if RUN_OBSERVATIONS:
    _ = run_demo("3b", experiment_3b, CHECK_LABELS)
else:
    LAB_RESULTS.pop("3b", None)
    print("[대기] 3(b): 세 함수 확인 후 관찰 실험 실행을 켜세요.")

### (c) 추가한 각도가 실제로 상대위상을 바꾸는지 확인하기

최적화 결과와 별도로, 새 매개변수 $\phi$가 회로에서 의도한 역할을 하는지 확인하겠습니다.

먼저

$$
\theta=\frac{\pi}{3}
$$

으로 고정하고, $\phi$를 다음 값들로 바꾸어 보세요.

$$
\phi=0,\quad \frac{\pi}{3},\quad \frac{\pi}{2},\quad \pi.
$$

이 $\theta$에서는 목표 상태의 두 성분이 모두 존재합니다. 제공된 결과 확인 실습으로 자신이 작성한 회로의 Z 측정 확률, $X_0Y_1$ 기댓값, 에너지를 확인하세요.

| $\phi$ | $P(00)$ | $P(11)$ | $\langle X_0Y_1\rangle$ | 에너지 |
|---|---:|---:|---:|---:|
| $0$ | … | … | … | … |
| $\pi/3$ | … | … | … | … |
| $\pi/2$ | … | … | … | … |
| $\pi$ | … | … | … | … |

다음으로, 아래 순서로 게이트를 적용하는 비교 회로를 살펴보세요.

1. 초기 상태 $|00\rangle$에서 q0에 $R_z(\phi)$를 적용합니다.
2. q0에 $R_y(\theta)$를 적용합니다.
3. $\mathrm{CX}(0,1)$을 적용합니다.

이 비교 회로도 같은 $\theta$와 네 $\phi$ 값으로 실행합니다. 회로에는 $R_z$가 들어 있지만, 그것이 여러분의 회로와 같은 역할을 하는지는 상태의 변화를 계산해서 판단해야 합니다.

두 실험을 바탕으로 다음을 설명하세요.

- <strong>자신의 회로:</strong> $\phi$를 바꾸었을 때 Z 측정 확률과 $X_0Y_1$ 기댓값은 각각 어떻게 달라지나요? 두 결과가 다르게 반응하는 이유를 목표 상태식과 연결하세요.
- <strong>비교 회로:</strong> 처음의 $R_z(\phi)$가 $|00\rangle$에 어떤 위상을 붙이는지 구하세요. 이후 게이트를 적용한 최종 상태에서 그 위상이 전역위상인지 상대위상인지 판단하세요.
- <strong>검증 방법:</strong> Z 측정 확률만 확인했다면 두 회로의 차이를 충분히 검증할 수 있었을까요? 이번에 추가로 확인한 관측량과 상태 비교가 어떤 정보를 주는지 설명하세요.

상태벡터의 복소 성분이 달라 보인다는 이유만으로 서로 다른 상태라고 판단하지 마세요. 상태 전체에 같은 위상이 곱해진 경우도 있으므로, <strong>두 성분 사이의 상대위상과 실제 관측 결과가 달라졌는지</strong>를 함께 확인하세요.

### 3c 답안

실행 전 예상과 계산:

…

실행 결과와 설명:

…

### 결과 확인 실습 3c

먼저 해당 문항의 계산과 예상을 적으세요. 필요한 코드 검사를 통과한 뒤 이 실험을 실행하고 실행 결과를 답안에 덧붙입니다. 아래 결과가 대기로 표시되면 앞의 실습 코드 실행 설정과 확인 셀을 점검하세요.

In [ ]:
def experiment_3c():
    theta = np.pi / 3
    results, rows = [], []
    for label, function in (("작성한 회로", phase_ansatz), ("초기 RZ 비교 회로", phase_before_rotation)):
        reference = circuit_state(function(theta, 0))
        for phi in (0, np.pi/3, np.pi/2, np.pi):
            circuit = function(theta, phi)
            state = circuit_state(circuit)
            probabilities = state.probabilities()
            values = expectations_from_circuit(circuit)
            energy = energy_from_expectations(values)
            fidelity = float(state_fidelity(reference, state))
            rows.append([label, phi, probabilities[0], probabilities[3], values["YX"], energy, fidelity])
            results.append({"circuit": label, "theta": theta, "phi": phi,
                            "probabilities": probabilities, "values": values,
                            "energy": float(energy), "fidelity_to_phi_zero": fidelity})
    show_table(["회로", "phi", "P(00)", "P(11)", "YX", "E", "같은 회로 phi=0과의 F"], rows)
    print("이 표의 fidelity 기준은 각 회로의 phi=0 상태이며, 바닥상태와의 비교가 아닙니다.")
    return results

if RUN_OBSERVATIONS:
    _ = run_demo("3c", experiment_3c, (CHECK_LABELS[0], CHECK_LABELS[1]))
else:
    LAB_RESULTS.pop("3c", None)
    print("[대기] 3(c): RZ 위치별 상태 변화를 예상한 뒤 관찰 실험을 실행하세요.")

## 문제 4. 최적화 결과를 어디까지 믿을 수 있을까요?

앞 문제에서는 상대위상을 조절하는 회로를 작성하고, 에너지 평가기를 고전 최적화기에 연결했습니다. 이제 <strong>최적화가 반환한 결과가 바닥상태 탐색의 성공을 뜻하는지</strong> 확인하겠습니다.

확인할 대상은 세 가지입니다. 첫째는 최적화기가 어떤 이유로 탐색을 끝냈는지, 둘째는 반환된 각도에서 에너지가 얼마나 낮은지, 셋째는 그 회로가 준비한 상태가 바닥상태와 얼마나 가까운지입니다.

이 세 정보는 서로 관련되어 있지만 같은 뜻은 아닙니다. 또한 측정 횟수가 유한하면, 같은 상태에서도 에너지 추정값이 달라질 수 있습니다. 이번 문제에서는 <strong>정확한 상태 계산으로 확인한 결과와 측정 기록으로 추정한 결과를 구분</strong>하여 해석하겠습니다.

---

### 종료 메시지는 탐색을 멈춘 이유를 알려 줍니다

고전 최적화기는 정해진 수치 조건에 따라 탐색을 끝냅니다. 예를 들어 현재 각도 주변에서 계산한 변화가 충분히 작다고 판단하거나, 허용된 계산 횟수에 도달하면 종료할 수 있습니다.

따라서 정상 종료 메시지가 나왔다고 해서, 가능한 모든 각도를 조사하여 가장 낮은 에너지를 찾았다는 뜻은 아닙니다. 반대로 종료 메시지에 수치적인 어려움이 표시되더라도, 반환된 각도의 상태가 이미 좋은 결과를 줄 수 있습니다.

종료 메시지는 실행 과정을 이해하는 데 사용하고, 결과의 성능은 별도로 계산해야 합니다. 이번에는 작은 두 큐비트 문제이므로 정확한 바닥 에너지와 바닥상태를 비교 기준으로 사용할 수 있습니다.

---

### 에너지가 기준에 얼마나 가까운지 확인하기

최적화기가 반환한 각도를 $(\theta_*,\phi_*)$라고 하겠습니다. 이 각도로 회로를 다시 준비하고, 정확한 에너지

$$
E_*
=
\langle\psi(\theta_*,\phi_*)|
\mathcal H
|\psi(\theta_*,\phi_*)\rangle
$$

를 계산합니다.

바닥 에너지와의 차이는

$$
\delta E=E_*-E_0
$$

로 기록하겠습니다. 이 값은 <strong>찾은 상태의 에너지가 정확한 바닥 에너지에서 얼마나 떨어져 있는지</strong> 나타냅니다.

변분 원리에 따르면 정규화된 상태의 정확한 에너지에서는

$$
\delta E\ge 0
$$

입니다. 수치 계산에서는 아주 작은 반올림 오차가 생길 수 있으므로, 결과를 비교할 때에는 정해진 허용 오차를 사용합니다. 화면에 반올림해 표시된 숫자만 보고 두 값이 같다고 판단하지 마세요.

이번 실습에서는 정확한 에너지 비교에

$$
|E_*-E_0|\le 10^{-6}
$$

을 기준으로 사용합니다. 이 기준은 에너지 계산과 상태 준비가 이상적인 이번 실습의 수치 검증 기준입니다.

---

### 에너지와 함께 준비된 상태도 확인하기

이번 Hamiltonian의 바닥상태는 하나입니다. 정확한 기준 바닥상태를 $|\psi_0\rangle$, 회로가 준비한 상태를 $|\psi_*\rangle$라고 하면 두 순수상태의 fidelity는

$$
F
=
|\langle\psi_0|\psi_*\rangle|^2
$$

입니다.

두 상태가 정규화되어 있을 때 $F$는 0과 1 사이의 값입니다. $F=1$이면 두 상태는 전역위상을 제외하고 같고, $F=0$이면 서로 직교합니다.

예를 들어 회로의 출력이

$$
|\psi_*\rangle=e^{i\chi}|\psi_0\rangle
$$

라면

$$
F
=
|e^{i\chi}\langle\psi_0|\psi_0\rangle|^2
=
1
$$

입니다. 상태벡터의 각 성분이 기준 벡터와 문자 그대로 같지 않더라도, 전체에 같은 위상이 붙은 경우에는 같은 물리적 상태로 판단합니다.

반면 계산 기저에서 같은 확률 분포가 나온다는 사실만으로 $F=1$이라고 할 수는 없습니다. 앞 문제에서 확인했듯이, 상대위상이 다른 상태들이 같은 Z 측정 확률을 줄 수 있기 때문입니다.

이번에는 제공 도구가 정확한 바닥상태와의 fidelity를 계산합니다. 상태 비교의 수치 기준은

$$
F\ge 1-10^{-6}
$$

으로 두겠습니다.

에너지 차이와 fidelity를 함께 보면, 낮은 에너지를 얻었다는 결과가 실제 상태의 준비와 일치하는지 확인할 수 있습니다. 기준 바닥상태는 이 검증 단계에서 사용하며, 최적화기의 목적함수에 정답 상태를 전달하지 않습니다.

---

### 정확한 에너지와 측정으로 추정한 에너지는 다릅니다

지금까지는 제공 도구로 정확한 기댓값을 계산했습니다. 이제 최적화한 각도를 고정하고, 유한한 횟수의 측정 기록으로 같은 상태의 에너지를 추정하겠습니다.

이번 표본 실험에서도 상태 준비와 게이트는 이상적입니다. 달라지는 것은 기댓값을 구하는 방법입니다. 정확한 확률을 직접 계산하는 대신, 각 설정을 정해진 횟수만큼 실행하고 관측값을 평균합니다.

Z 설정의 기록에서는

$$
\widehat{\langle Z_0Z_1\rangle},
\qquad
\widehat{\langle Z_0\rangle},
\qquad
\widehat{\langle Z_1\rangle}
$$

을 구하고, 별도로 준비한 상태의 XY 설정 기록에서는

$$
\widehat{\langle X_0Y_1\rangle}
$$

을 구합니다. 이를 합친 에너지 추정값은

$$
\widehat E
=
-\widehat{\langle Z_0Z_1\rangle}
-\widehat{\langle X_0Y_1\rangle}
-\frac12\widehat{\langle Z_0\rangle}
-\frac12\widehat{\langle Z_1\rangle}
$$

입니다.

같은 각도를 사용해도 측정 기록은 달라질 수 있으므로, 반복해서 계산한 $\widehat E$도 조금씩 달라질 수 있습니다. 이때 <strong>회로가 준비하는 상태가 매번 더 좋아지거나 나빠진 것은 아닙니다.</strong> 같은 상태의 에너지를 유한한 기록으로 추정하면서 변동이 생긴 것입니다.

변분 원리의 $E\ge E_0$는 정규화된 상태의 <strong>정확한 에너지 기댓값</strong>에 적용됩니다. 서로 다른 측정 설정의 유한한 표본 평균들을 합한 $\widehat E$가 매번 이 부등식을 만족해야 하는 것은 아닙니다. 따라서 추정값 하나가 $E_0$보다 낮게 나왔을 때에는, 정확한 기댓값과 표본 추정값을 구분해서 해석해야 합니다.

---

### 측정 횟수를 늘리면 무엇이 달라질까요?

제공된 실험에서는 앞 문제에서 찾은 각도를 고정하고, 설정별 shots를 바꾸어 에너지를 반복 추정합니다.

| 설정별 shots $N$ | 에너지 추정 한 번에 사용하는 전체 shots |
|---:|---:|
| 64 | $2N$ |
| 256 | $2N$ |
| 1,024 | $2N$ |

각 에너지 추정에는 Z 설정 $N$회와 XY 설정 $N$회가 사용됩니다. Z 관련 세 기댓값은 같은 Z 설정 기록에서 구합니다.

각 shots 조건에서 에너지 추정을 20번 반복하고, 다음 정보를 표로 제공합니다.

- 반복 추정값의 평균
- 추정값들이 퍼진 정도를 나타내는 표준편차
- 가장 낮게 나온 추정값과 가장 높게 나온 추정값
- 비교 기준이 되는 정확한 에너지

여기서 표준편차는 <strong>같은 상태의 에너지를 반복 추정했을 때 값이 얼마나 달라졌는지</strong>를 나타냅니다.

Shots를 늘리면 일반적으로 표본 변동은 줄어들지만, 한 번씩 얻은 결과가 반드시 정확값에 더 가까워지는 순서로 나올 필요는 없습니다. 따라서 한 번의 추정값만 비교하기보다 반복 결과의 분포를 함께 살펴보겠습니다.

---

### 결과가 좋지 않을 때 무엇을 바꿔야 할까요?

에너지가 충분히 낮아지지 않은 경우에도 원인은 다를 수 있습니다.

회로가 필요한 상태를 준비할 수 없다면, 그 회로 안에서 각도를 더 정밀하게 탐색해도 한계를 넘을 수 없습니다. 반면 같은 회로가 다른 각도에서는 좋은 상태를 준비할 수 있다면, 초기값이나 탐색 과정을 살펴볼 필요가 있습니다.

정확한 에너지는 충분히 낮지만 표본 추정값이 크게 흔들린다면, 먼저 측정 횟수와 추정의 변동을 확인해야 합니다.

이 원인들을 구분하려면 결과 숫자 하나보다 <strong>사용한 회로, 초기값, 에너지 계산 방법, 비교 기준</strong>을 함께 기록해야 합니다. 다음 문항에서는 앞에서 얻은 결과를 이 관점에서 검토하겠습니다.

---

### 직접 풀어보기

### (a) 수정 전후의 상태를 정확한 기준과 비교하기

다음 두 상태를 비교하세요.

- 문제 2에서 구한, 첫 회로의 최소 에너지를 달성하는 상태
- 문제 3의 최적화기가 반환한 각도로 확장 회로가 준비한 상태

제공된 검증 도구로 정확한 에너지와 바닥상태에 대한 fidelity를 계산하여 표를 완성하세요.

| 비교 대상 | 사용한 각도 | 정확한 에너지 $E$ | $E-E_0$ | 바닥상태와의 fidelity |
|---|---|---:|---:|---:|
| 첫 회로에서 찾은 최적 상태 | … | … | … | … |
| 확장 회로의 최적화 결과 | … | … | … | … |

이어서 다음을 설명하세요.

1. 각 상태가 이번 실습의 에너지 기준과 fidelity 기준을 만족하나요?
2. 회로를 확장한 뒤 에너지와 상태 일치도가 각각 어떻게 달라졌나요?
3. 최적화기의 종료 메시지와 별도로 이 값을 계산해야 하는 이유는 무엇인가요?

검증 결과를 적을 때에는 최적화기가 사용한 에너지 기록만 옮기지 말고, <strong>반환된 각도로 회로를 다시 준비해 계산한 결과</strong>를 사용하세요.

---

### 4a 답안

계산·실행 결과와 근거:

…

### 결과 확인 실습 4a

먼저 해당 문항의 계산과 예상을 적으세요. 필요한 코드 검사를 통과한 뒤 이 실험을 실행하고 실행 결과를 답안에 덧붙입니다. 아래 결과가 대기로 표시되면 앞의 실습 코드 실행 설정과 확인 셀을 점검하세요.

In [ ]:
def experiment_4a():
    if "2bc" not in LAB_RESULTS or "3b" not in LAB_RESULTS:
        raise ValueError("먼저 2(b,c) 스캔과 3(b) 최적화 실험을 실행하세요.")
    first_theta = LAB_RESULTS["2bc"]["best"]["theta"]
    expanded = LAB_RESULTS["3b"]["parameters"]
    results, rows = [], []
    for label, parameters, circuit in (
        ("첫 회로: 스캔에서 선택", (first_theta,), real_ansatz(first_theta)),
        ("확장 회로: 최적화 반환", expanded, phase_ansatz(*expanded)),
    ):
        report = validate_result(circuit)
        results.append({"label": label, "parameters": parameters, **report})
        rows.append([label, parameters, report["energy"], report["delta_energy"], report["fidelity"], report["energy_pass"], report["state_pass"]])
    show_table(["비교 대상", "각도", "정확한 E", "E-E0", "바닥상태 F", "에너지 기준", "상태 기준"], rows)
    print("첫 행의 각도가 수식으로 구한 최소점인지 문제 2의 답안과 함께 확인하세요.")
    return results

if RUN_OBSERVATIONS:
    _ = run_demo("4a", experiment_4a, CHECK_LABELS)
else:
    LAB_RESULTS.pop("4a", None)
    print("[대기] 4(a): 앞의 실험 후 정확한 결과를 비교하세요.")

### (b) 같은 상태의 에너지를 유한 shots로 추정하기

문제 3에서 찾은 각도를 고정한 채 제공된 표본 실험을 실행하세요. 이 문항에서는 측정 결과에 따라 각도를 다시 최적화하지 않습니다.

각 shots 조건의 반복 결과를 정리하세요.

| 설정별 shots | 정확한 에너지 | 추정값의 평균 | 추정값의 표준편차 | 가장 낮은 추정값 | 가장 높은 추정값 |
|---:|---:|---:|---:|---:|---:|
| 64 | … | … | … | … | … |
| 256 | … | … | … | … | … |
| 1,024 | … | … | … | … | … |

다음 질문에 답하세요.

- Shots가 달라졌을 때 추정값의 퍼짐은 어떻게 달라졌나요? 한 번의 결과와 반복 결과에서 보이는 경향을 구분해서 설명하세요.
- $E_0$보다 낮은 추정값이 있다면, 그 기록이 변분 원리를 위반한 상태를 준비했다는 뜻인가요? 없다면, 다른 측정 기록에서도 그런 값이 절대 나올 수 없다고 결론 내릴 수 있나요?
- 같은 각도로 얻은 20개 추정값 중 가장 낮은 값만 골라 “이 상태의 에너지”로 보고하면 어떤 문제가 있나요?

마지막으로, 측정으로 얻은 결과를 보고할 때 <strong>추정값 하나 외에 어떤 정보를 함께 적어야 하는지</strong> 정리하세요.

---

### 4b 답안

실행 전 예상과 계산:

…

실행 결과와 설명:

…

### 결과 확인 실습 4b

먼저 해당 문항의 계산과 예상을 적으세요. 필요한 코드 검사를 통과한 뒤 이 실험을 실행하고 실행 결과를 답안에 덧붙입니다. 아래 결과가 대기로 표시되면 앞의 실습 코드 실행 설정과 확인 셀을 점검하세요.

In [ ]:
def experiment_4b():
    if "3b" not in LAB_RESULTS:
        raise ValueError("먼저 3(b) 최적화 실험을 실행하세요.")
    parameters = LAB_RESULTS["3b"]["parameters"]
    circuit = phase_ansatz(*parameters)
    reports = repeated_energy_estimates(circuit, energy_from_expectations,
                                        shots_values=(64, 256, 1024), repeats=20, seed=20261109)
    reference = ground_reference()["energy"]
    show_table(["설정별 shots", "정확한 E", "20회 평균", "표본 표준편차", "최저 추정", "최고 추정", "전체 shots / 추정"],
               [[r["shots_per_setting"], r["exact_energy"], r["mean"], r["sample_std"], r["min"], r["max"], r["total_shots_per_estimate"]] for r in reports])
    show_table(["추정 순서", "N=64", "N=256", "N=1024"],
               [[i+1, *[r["energies"][i] for r in reports]] for i in range(20)])
    show_table(["설정별 shots", "E0 미만 추정 횟수", "이 조건의 총 shots"],
               [[r["shots_per_setting"], sum(value < reference for value in r["energies"]), r["total_shots"]] for r in reports])
    print("모든 표본 실험에서 각도를 고정했습니다:", parameters)
    print("Z 기록의 세 통계는 같은 측정 기록을 사용합니다. 각 에너지 추정의 Z/XY 표본은 별도로 생성합니다.")
    return {"parameters": parameters, "reports": reports}

if RUN_OBSERVATIONS:
    _ = run_demo("4b", experiment_4b, CHECK_LABELS)
else:
    LAB_RESULTS.pop("4b", None)
    print("[대기] 4(b): 각도를 고정한 측정 추정 실험입니다. 관찰 실험 실행을 켜세요.")

### (c) 결과 보고서를 보고 다음 조치 정하기

제공된 비교 실험에서는 확장 회로와 목적함수를 유지하고, 최적화 초기값을

$$
(0.4,0.6)
\quad\text{또는}\quad
(0,0)
$$

으로 바꾸어 실행합니다. 각 실행의 종료 메시지, 반환된 각도, 정확한 에너지를 앞의 결과와 함께 확인하세요.

이어서 다음 세 보고서를 검토합니다. 각 보고서에서 확인된 사실과 아직 확인되지 않은 주장을 구분하고, 다음에 무엇을 해야 할지 제안하세요.

> <strong>보고서 A</strong>  
> “첫 회로에서 여러 각도를 시험했지만 바닥 에너지에 도달하지 못했다. 각도 간격을 더 촘촘하게 하고 최적화 시간을 늘리면 해결될 것이다.”

문제 2에서 구한 에너지 식과 최소 에너지를 근거로 판단하세요. 추가 탐색이 해결할 수 있는 문제인지 설명합니다.

> <strong>보고서 B</strong>  
> “확장 회로를 한 초기값에서 실행했더니 최적화기가 정상 종료했다. 따라서 이 회로에서 가능한 가장 낮은 에너지를 찾았다.”

같은 회로를 다른 초기값에서 실행한 결과와 비교하세요. 정상 종료라는 사실만으로 가능한 결론이 무엇인지, 더 낮은 에너지를 주는 각도가 발견되었다면 보고서의 주장을 어떻게 수정해야 하는지 설명합니다.

> <strong>보고서 C</strong>  
> “같은 각도에서 측정한 에너지 추정값이 실행할 때마다 달랐다. 따라서 회로가 매번 서로 다른 품질의 상태를 준비한 것으로 보인다.”

이번 실험의 상태 준비 조건과 에너지 추정 방법을 근거로 판단하세요. 상태의 변화와 추정값의 변동을 구분해서 설명합니다.

각 보고서의 마지막에는 다음 조치 중 필요한 것을 고르고 이유를 적으세요. 여러 조치가 필요하다면 순서도 제안할 수 있습니다.

- 회로가 준비할 수 있는 상태의 범위 확인
- 회로 구조 수정
- 초기값이나 탐색 조건 변경
- 정확한 에너지 재계산
- 측정 횟수 증가와 반복 추정
- 결과 보고 방식 수정

조치를 선택할 때에는 “값이 좋지 않다”는 이유만 쓰지 말고, <strong>어떤 결과를 근거로 그 조치가 필요하다고 판단했는지</strong> 연결해서 설명하세요.

---

### 4c 답안

실행 전 예상과 계산:

…

실행 결과와 설명:

…

### 결과 확인 실습 4c

먼저 해당 문항의 계산과 예상을 적으세요. 필요한 코드 검사를 통과한 뒤 이 실험을 실행하고 실행 결과를 답안에 덧붙입니다. 아래 결과가 대기로 표시되면 앞의 실습 코드 실행 설정과 확인 셀을 점검하세요.

In [ ]:
def experiment_4c():
    if "3b" not in LAB_RESULTS:
        raise ValueError("먼저 3(b) 최적화 실험을 실행하세요.")
    reports = [LAB_RESULTS["3b"], optimize_vqe(vqe_objective, initial=(0.0, 0.0))]
    results, rows = [], []
    for report in reports:
        validation = validate_result(phase_ansatz(*report["parameters"]))
        results.append({"optimizer": report, "validation": validation})
        rows.append([report["initial"], report["parameters"], report["success"],
                     validation["energy"], validation["delta_energy"], validation["fidelity"], report["nfev"], report["nit"]])
    show_table(["초기값", "반환 각도", "정상 종료", "정확한 E", "E-E0", "바닥상태 F", "평가 횟수", "반복 횟수"], rows)
    for report in reports:
        print("초기값", report["initial"], "종료 메시지:", report["message"])
    print("동일한 회로와 현재 BFGS 설정에서 얻은 실제 기록입니다. 모든 최적화기의 일반적인 동작으로 해석하지 않습니다.")
    return results

if RUN_OBSERVATIONS:
    _ = run_demo("4c", experiment_4c, CHECK_LABELS)
else:
    LAB_RESULTS.pop("4c", None)
    print("[대기] 4(c): 앞의 실험 결과와 초기값 비교를 함께 검토하세요.")

## 마지막 적용 — 에너지 항 하나의 부호가 바뀐다면?

이제 회로 구조와 측정 방법을 유지한 채, Hamiltonian의 $X_0Y_1$ 항의 계수만 바꾸겠습니다.

$$
\mathcal H'
=
-Z_0Z_1+X_0Y_1-\frac12(Z_0+Z_1).
$$

앞에서 찾은 각도를 그대로 사용하면 같은 상태가 준비됩니다. 그러나 상태를 평가하는 Hamiltonian이 달라졌으므로, 그 상태의 에너지도 다시 계산해야 합니다.

새 최적화를 실행하기 전에 다음 순서로 예상과 수정 방법을 적으세요.

1. <strong>기존 상태의 새 에너지 예상하기</strong>  
   앞에서 얻은 네 기댓값을 사용하여, 기존 각도를 그대로 썼을 때 $\mathcal H'$에 대한 에너지를 예상하세요. 어떤 항의 기여가 바뀌는지 식으로 나타냅니다.

2. <strong>매개변수 수정 제안하기</strong>  
   회로 구조를 유지하면서 $\theta,\phi$를 어떻게 바꾸면 좋을지 제안하세요. 바뀐 에너지 항과 상태의 관계를 근거로 설명합니다.

3. <strong>예상과 수정 결과 확인하기</strong>  
   앞에서 작성한 에너지 함수에 `xy_coeff=+1.0`을 전달하여 새 에너지를 계산하세요. 이어서 제공된 기준 계산으로 새 Hamiltonian의 바닥 에너지와 바닥상태를 구하고, 수정한 상태를 검증합니다.

| 준비한 상태 | 사용한 각도 | 예상한 새 에너지 | 계산한 새 에너지 | 새 바닥 에너지와의 차이 | 새 바닥상태와의 fidelity |
|---|---|---:|---:|---:|---:|
| 기존 각도를 그대로 사용 | … | … | … | … | … |
| 제안한 각도로 수정 | … | … | … | … | … |

비교 기준도 새 Hamiltonian에 맞추어야 합니다. Fidelity는 원래 문제의 바닥상태가 아니라 <strong>$\mathcal H'$의 바닥상태</strong>에 대해 계산하세요.

새 회로나 최적화기를 작성할 필요는 없습니다. 마지막으로 <strong>Hamiltonian이 바뀌었을 때 유지한 도구와 수정한 선택이 무엇인지</strong>, 그리고 그 수정이 왜 적절했는지 짧게 정리하세요.

## 마지막 적용 답안

실행 전 예상과 계산:

…

실행 결과와 설명:

…

### 적용 결과 확인

기존 상태의 예상 새 에너지는 `PREDICTED_NEW_ENERGY`에, 제안한 두 각도는 `APPLICATION_PARAMETERS`에 `(theta, phi)` 순서로 적으세요. 그런 다음 `RUN_APPLICATION=True`로 바꾸고 아래 실험을 실행합니다. 수정한 상태의 예상 에너지도 위의 서술 답안 표에 먼저 적어 두세요. 이 변수들은 결과 확인 실습의 입력이며 새 함수 TODO가 아닙니다.

In [ ]:
RUN_APPLICATION = False
APPLICATION_PARAMETERS = None
PREDICTED_NEW_ENERGY = None

In [ ]:
def application_experiment():
    if "3b" not in LAB_RESULTS:
        raise ValueError("먼저 3(b) 최적화 실험을 실행하세요.")
    if APPLICATION_PARAMETERS is None or PREDICTED_NEW_ENERGY is None:
        raise ValueError("먼저 제안할 두 각도와 기존 상태의 예상 새 에너지를 입력하세요.")
    proposed = tuple(map(float, APPLICATION_PARAMETERS))
    if len(proposed) != 2 or not np.isfinite(proposed).all():
        raise ValueError("제안 각도는 유한한 theta,phi 두 값입니다.")
    predicted = float(PREDICTED_NEW_ENERGY)
    if not np.isfinite(predicted):
        raise ValueError("예상 에너지는 유한한 실수입니다.")
    baseline = LAB_RESULTS["3b"]["parameters"]
    results, rows = [], []
    for label, parameters in (("기존 각도 유지", baseline), ("학생이 제안한 각도", proposed)):
        circuit = phase_ansatz(*parameters)
        values = expectations_from_circuit(circuit)
        energy = float(energy_from_expectations(values, xy_coeff=1.0))
        validation = validate_result(circuit, xy_coeff=1.0)
        assert np.isclose(energy, validation["energy"], atol=ATOL, rtol=0), "새 Hamiltonian의 계수와 평가기를 확인하세요."
        rows.append([label, parameters, energy, validation["delta_energy"], validation["fidelity"]])
        results.append({"label": label, "parameters": parameters, "values": values, **validation})
    show_table(["준비한 상태", "각도", "새 H의 E", "E-E0'", "새 바닥상태 F"], rows)
    matches = bool(np.isclose(predicted, results[0]["energy"], atol=ENERGY_TOL, rtol=0))
    show_table(["기존 상태의 예상 새 에너지", "계산한 새 에너지", "예상과 일치"],
               [[predicted, results[0]["energy"], matches]])
    print("새 Hamiltonian의 정확한 E0':", results[0]["reference_energy"])
    print("비교 기준 상태도 새 Hamiltonian에서 다시 계산했습니다.")
    return {"rows": results, "predicted_new_energy": predicted, "prediction_matches": matches}

LAB_RESULTS.pop("application", None)
if RUN_APPLICATION:
    _ = run_demo("application", application_experiment, CHECK_LABELS)
else:
    print("[대기] 마지막 적용: 예상과 제안 각도를 먼저 적은 뒤 RUN_APPLICATION을 True로 바꾸세요.")

## Lab 4 마무리

세 코드 검사의 상태와 각 소문항의 설명을 확인하세요. 마지막으로 에너지 평가의 정확성, 회로가 표현할 수 있는 상태, 최적화기의 탐색, 유한한 측정 기록의 추정을 구분하여 세 문장 또는 네 문장으로 정리하세요. 결과가 충분히 좋지 않을 때 어떤 근거로 다음 조치를 정할지도 포함합니다.

### 마무리 답안

계산·실행 결과와 근거:

…

In [ ]:
expected_checks = ['1b · 에너지 평가기', '3a · 상대위상 회로', '3b · 최적화 목적함수']
show_table(["코드 확인", "상태"], [[name, CHECK_STATUS.get(name, "미실행")] for name in expected_checks])
if all(CHECK_STATUS.get(name) == "통과" for name in expected_checks):
    print("세 코드 검사를 모두 통과했습니다. 서술 답안과 마지막 적용 해석은 별도로 확인합니다.")
else:
    print("TODO와 확인 셀을 작성·실행한 뒤 연결 실험도 다시 실행하세요.")